In [ ]:
# Configuração do ambiente (Colab ou local)
import os
import sys
import urllib.request
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PUBLIC_REPO_RAW = "https://raw.githubusercontent.com/vdms/mvp-machine-learning/main"
PUBLIC_CACHE_BASE_URL = f"{PUBLIC_REPO_RAW}/outputs/"
PUBLIC_NOTEBOOK_URL = f"{PUBLIC_REPO_RAW}/notebooks/mvp_ml_analytics_enem_rio.ipynb"
CACHE_FILENAMES = [
    "enem_rj_results_2022_2024.csv.gz",
    "enem_rj_profiles_2022_2024.csv.gz",
]
DATA_SOURCE_MODE = "public_cache" if IN_COLAB else "local_or_official_zip"

if IN_COLAB:
    Path("outputs").mkdir(parents=True, exist_ok=True)
    for cache_name in CACHE_FILENAMES:
        destination = Path("outputs") / cache_name
        if destination.exists() and destination.stat().st_size > 0:
            print(f"Cache encontrado: {destination}")
            continue
        url = PUBLIC_CACHE_BASE_URL + cache_name
        print(f"Baixando {cache_name}...")
        try:
            urllib.request.urlretrieve(url, destination)
        except Exception as exc:
            raise RuntimeError(f"Falha ao baixar cache publico: {url}") from exc
        if not destination.exists() or destination.stat().st_size == 0:
            raise RuntimeError(f"Cache baixado vazio ou ausente: {destination}")
        print(f"  OK - {destination.stat().st_size / 1e6:.1f} MB")


# MVP Machine Learning & Analytics - ENEM Rio/RJ 2022-2024

Este notebook é um relatório técnico executável. Ele testa se variáveis socioeducacionais, escolares e contextuais ajudam a prever alto desempenho em Matemática no ENEM/RJ e se parte desse padrão permanece útil em 2024.

A entrega combina dois experimentos complementares. O Modelo A responde à pergunta socioeducacional individual em 2022-2023. O Modelo B responde à pergunta de robustez temporal em 2024 usando variáveis comparáveis e agregados municipais. Os experimentos não são uma competição direta entre algoritmos.


## Contexto e motivação

O ENEM é uma avaliação nacional de grande escala e permite investigar desigualdades educacionais com dados públicos. O foco aqui não é usar notas ou respostas para prever a própria nota, mas testar o quanto características disponíveis antes ou fora do resultado da prova carregam sinal preditivo.

Essa escolha torna o MVP relevante para uma discussão educacional: desempenho alto não é tratado apenas como número, mas como fenômeno associado a trajetória escolar, perfil socioeconômico e contexto territorial.


## Definição do problema

**Problema:** classificar participantes do ENEM/RJ com alto desempenho em Matemática.

**Target:** `target_desempenho_alto_mt = 1` quando `NU_NOTA_MT` é maior ou igual ao percentil 75 calculado apenas no treino do respectivo experimento.

**Por que é Machine Learning:** há uma variável-alvo observada, múltiplos atributos explicativos e avaliação em dados não vistos. Como as classes são desbalanceadas, o F1-score é a métrica principal, com accuracy, precision, recall, ROC-AUC e matriz de confusão como métricas complementares.


## Hipóteses do Estudo

**H1:** variáveis socioeducacionais e escolares possuem capacidade preditiva relevante para identificar estudantes com alto desempenho em Matemática no ENEM.

**H2:** parte dos padrões aprendidos a partir dos dados de 2022-2023 permanece estável quando avaliada em dados de 2024.

**H3:** modelos supervisionados devem superar um baseline ingênuo tanto na avaliação convencional quanto na avaliação temporal.

As hipóteses são testáveis por comparação com `DummyClassifier`, avaliação em dados não vistos, F1-score, ROC-AUC, matriz de confusão e teste externo em 2024.


## Fonte dos dados e reprodutibilidade

Fonte oficial: [Microdados do ENEM - INEP](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem).

Em execução local, o notebook reaproveita bases derivadas em `outputs/` ou processa os ZIPs oficiais em `data/raw/`. No Google Colab, para evitar baixar e processar cerca de 1,8 GB de ZIPs brutos durante a avaliação, o notebook baixa automaticamente duas bases públicas derivadas dos microdados oficiais: resultados elegíveis e perfis do recorte RJ. Os microdados brutos não devem ser versionados; apenas artefatos resumidos e bases derivadas leves são salvos em `artifacts/` e `outputs/`.


## Ambiente, bibliotecas e constantes

Esta célula centraliza imports, diretórios, seeds, URLs oficiais e listas de variáveis. As constantes deixam a execução rastreável e reduzem decisões escondidas no código.


In [ ]:
from __future__ import annotations

import ast
import csv
import json
import os
import re
import time
import urllib.request
import zipfile
from pathlib import Path

_CURRENT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_MPLCONFIGDIR = _CURRENT_ROOT / "outputs" / ".matplotlib"
_MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(_MPLCONFIGDIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
ARTIFACTS = PROJECT_ROOT / "artifacts"
OUTPUTS = PROJECT_ROOT / "outputs"
for directory in [DATA_RAW, ARTIFACTS, OUTPUTS]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED_YEARS = {2022, 2023, 2024}
TRAIN_YEARS = {2022, 2023}
EXTERNAL_TEST_YEAR = 2024
RECORTE_UF = "RJ"
TARGET_SCORE_COLUMN = "NU_NOTA_MT"
TARGET_COLUMN = "target_desempenho_alto_mt"
TARGET_QUANTILE = 0.75
CHUNKSIZE = 200_000
MAX_TRAIN_ROWS_FOR_MODELING = 60_000
RF_SEARCH_N_ESTIMATORS = 80
RF_FINAL_N_ESTIMATORS = 120
MODEL_SELECTION_FOLDS = 3
MODEL_PARALLEL_JOBS = min(2, os.cpu_count() or 1)
MODEL_SELECTION_PRE_DISPATCH = 1
MODEL_A_TIE_MARGIN = 0.005
MODEL_B_TIE_MARGIN = 0.010
F1_BOOTSTRAP_REPETITIONS = 1_000
# Em Colab, usar bases derivadas publicas para evitar baixar/processar os ZIPs brutos.
DOWNLOAD_IF_MISSING = not IN_COLAB

ENEM_DOWNLOAD_URLS = {
    2022: "https://download.inep.gov.br/microdados/microdados_enem_2022.zip",
    2023: "https://download.inep.gov.br/microdados/microdados_enem_2023.zip",
    2024: "https://download.inep.gov.br/microdados/microdados_enem_2024.zip",
}

QUESTIONARIO_COLUMNS_2022_2023 = [f"Q{i:03d}" for i in range(1, 26)]
QUESTIONARIO_COLUMNS_2024 = [f"Q{i:03d}" for i in range(1, 24)]
PROFILE_COLUMNS = [
    "TP_FAIXA_ETARIA", "TP_SEXO", "TP_ESTADO_CIVIL", "TP_COR_RACA", "TP_NACIONALIDADE",
    "TP_ST_CONCLUSAO", "TP_ANO_CONCLUIU", "TP_ESCOLA", "TP_ENSINO", "IN_TREINEIRO",
]
SCHOOL_COLUMNS = [
    "CO_MUNICIPIO_ESC", "NO_MUNICIPIO_ESC", "CO_UF_ESC", "SG_UF_ESC",
    "TP_DEPENDENCIA_ADM_ESC", "TP_LOCALIZACAO_ESC", "TP_SIT_FUNC_ESC",
]
PROVA_COLUMNS = ["CO_MUNICIPIO_PROVA", "NO_MUNICIPIO_PROVA", "CO_UF_PROVA", "SG_UF_PROVA"]
PRESENCE_COLUMNS = ["TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT"]
SCORE_COLUMNS = ["NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO"]
REDACAO_COMP_COLUMNS = [f"NU_NOTA_COMP{i}" for i in range(1, 6)]
LEAKAGE_PREFIXES = ("NU_NOTA", "TX_RESPOSTAS", "TX_GABARITO", "CO_PROVA")
AGGREGATE_PROFILE_COLUMNS = ["Q006", "TP_COR_RACA", "TP_SEXO", "TP_FAIXA_ETARIA", "IN_TREINEIRO", "TP_ST_CONCLUSAO", "TP_ENSINO"]

REQUESTED_2022_2023_COLUMNS = (
    ["NU_INSCRICAO", "NU_ANO"] + PROFILE_COLUMNS + SCHOOL_COLUMNS + PROVA_COLUMNS +
    PRESENCE_COLUMNS + SCORE_COLUMNS + REDACAO_COMP_COLUMNS + QUESTIONARIO_COLUMNS_2022_2023
)
REQUESTED_2024_RESULT_COLUMNS = ["NU_ANO"] + SCHOOL_COLUMNS + PROVA_COLUMNS + PRESENCE_COLUMNS + SCORE_COLUMNS + REDACAO_COMP_COLUMNS
REQUESTED_2024_PARTICIPANT_COLUMNS = ["NU_INSCRICAO", "NU_ANO"] + [c for c in PROFILE_COLUMNS if c != "TP_ESCOLA"] + PROVA_COLUMNS + QUESTIONARIO_COLUMNS_2024

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 80)
CHART_COLORS = {
    "blue": "#4E79A7",
    "orange": "#F28E2B",
    "green": "#59A14F",
    "red": "#C44E52",
    "gray": "#9CA3AF",
    "dark_gray": "#374151",
}
YEAR_COLORS = {2022: CHART_COLORS["blue"], 2023: CHART_COLORS["orange"], 2024: CHART_COLORS["green"]}
MODEL_COLORS = {
    "DummyClassifier": CHART_COLORS["gray"],
    "LogisticRegression": CHART_COLORS["blue"],
    "RandomForestClassifier": CHART_COLORS["green"],
    "RandomForestClassifier_Optimized": CHART_COLORS["orange"],
}

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=[CHART_COLORS["blue"], CHART_COLORS["orange"], CHART_COLORS["green"], CHART_COLORS["red"], CHART_COLORS["gray"]]),
})

assert EXPECTED_YEARS == set(ENEM_DOWNLOAD_URLS)
assert TRAIN_YEARS.isdisjoint({EXTERNAL_TEST_YEAR})


### Nota sobre a otimização do tempo de execução

Nos primeiros testes no Google Colab, o notebook levava mais de 20 minutos para executar. A maior parte desse tempo era usada no treinamento repetido da Random Forest, na busca de hiperparâmetros e na validação cruzada.

Para tentar respeitar o limite de 15 minutos definido para o trabalho, reorganizei essa parte do notebook. A escolha dos modelos e dos hiperparâmetros continua sendo feita apenas com os dados de treino. A busca da Random Forest usa uma amostra estratificada, enquanto a regressão logística final usa todo o conjunto de treino. Também evitei repetir treinamentos, transformações e previsões que já tinham sido calculados.

Usei ferramentas baseadas em LLMs como apoio para revisar o código, identificar repetições e estudar alternativas para reduzir o tempo de execução. As decisões foram conferidas com testes controlados, e a responsabilidade pela execução, pelas escolhas metodológicas e pela interpretação dos resultados continua sendo minha.

Na validação local com as bases derivadas já disponíveis, a execução completa ficou em aproximadamente 68 segundos. Esse resultado não substitui a medição em uma sessão nova do Colab, onde o limite continua sendo 15 minutos.


## Funções de leitura, download e validação

As funções abaixo localizam os ZIPs oficiais, fazem download se necessário, escolhem o CSV correto em cada ano e leem os arquivos em chunks. A leitura em chunks é necessária porque os microdados são grandes.


In [ ]:
def find_enem_files(raw_dir: Path = DATA_RAW) -> list[Path]:
    """Retorna os arquivos brutos locais do ENEM ignorando marcadores vazios."""
    files = sorted(raw_dir.glob("*.zip")) + sorted(raw_dir.glob("*.csv"))
    return [path for path in files if path.name != ".gitkeep"]


In [ ]:
def infer_year_from_name(path: Path) -> int | None:
    """Extrai o ano do ENEM a partir do nome de um arquivo."""
    match = re.search(r"20\d{2}", path.name)
    return int(match.group(0)) if match else None


In [ ]:
def years_available_locally(files: list[Path]) -> set[int]:
    """Lista os anos que já possuem arquivo local disponível."""
    return {year for path in files if (year := infer_year_from_name(path)) is not None}


In [ ]:
def download_file(url: str, destination: Path) -> None:
    """Baixa um arquivo remoto para o destino informado com escrita temporária segura."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp_destination = destination.with_suffix(destination.suffix + ".partial")
    print(f"Baixando {url} -> {destination.relative_to(PROJECT_ROOT)}")
    with urllib.request.urlopen(url) as response, temp_destination.open("wb") as out_file:
        total = int(response.headers.get("Content-Length", "0"))
        downloaded = 0
        next_report = 0
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            out_file.write(chunk)
            downloaded += len(chunk)
            if total and downloaded >= next_report:
                pct = downloaded / total * 100
                print(f"  {pct:5.1f}% ({downloaded / (1024 ** 2):.1f} MB)")
                next_report += max(total // 10, 1)
    temp_destination.replace(destination)


In [ ]:
def ensure_enem_files_available() -> list[Path]:
    """Garante que os arquivos dos anos esperados existam localmente ou sejam baixados."""
    files = find_enem_files()
    missing_years = sorted(EXPECTED_YEARS - years_available_locally(files))
    if missing_years and DOWNLOAD_IF_MISSING:
        for year in missing_years:
            destination = DATA_RAW / f"microdados_enem_{year}.zip"
            if not destination.exists():
                download_file(ENEM_DOWNLOAD_URLS[year], destination)
        files = find_enem_files()
        missing_years = sorted(EXPECTED_YEARS - years_available_locally(files))
    if missing_years:
        raise FileNotFoundError(f"Arquivos ausentes para os anos {missing_years}.")
    return sorted([path for path in files if infer_year_from_name(path) in EXPECTED_YEARS], key=lambda p: infer_year_from_name(p) or 0)


In [ ]:
def list_csv_members(zip_path: Path) -> list[str]:
    """Lista os arquivos CSV internos de um ZIP de microdados."""
    with zipfile.ZipFile(zip_path) as zf:
        return [name for name in zf.namelist() if name.lower().endswith(".csv")]


In [ ]:
def select_csv_member(zip_path: Path, kind: str) -> str:
    """Seleciona o CSV interno apropriado para microdados, resultados ou participantes."""
    members = list_csv_members(zip_path)
    year = infer_year_from_name(zip_path)
    if kind == "microdados":
        candidates = [m for m in members if "microdados" in Path(m).name.lower()]
    elif kind == "resultados":
        candidates = [m for m in members if "resultados" in Path(m).name.lower()]
        if not candidates and year in {2022, 2023}:
            candidates = [m for m in members if "microdados" in Path(m).name.lower()]
    elif kind == "participantes":
        candidates = [m for m in members if "participantes" in Path(m).name.lower()]
        if not candidates and year in {2022, 2023}:
            candidates = [m for m in members if "microdados" in Path(m).name.lower()]
    else:
        raise ValueError(f"Tipo de CSV desconhecido: {kind}")
    if not candidates:
        raise FileNotFoundError(f"Nao encontrei CSV do tipo {kind} em {zip_path.name}.")
    return candidates[0]


In [ ]:
def sniff_delimiter_from_text(sample: str) -> str:
    """Detecta o delimitador provável em uma amostra textual de CSV."""
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=";,")
        return dialect.delimiter
    except csv.Error:
        return ";"


In [ ]:
def read_sample_text(path: Path, member: str, n_bytes: int = 100_000) -> str:
    """Lê uma pequena amostra textual de um CSV dentro de um ZIP."""
    with zipfile.ZipFile(path) as zf:
        return zf.open(member).read(n_bytes).decode("latin1", errors="replace")


In [ ]:
def read_header(path: Path, member: str) -> list[str]:
    """Lê o cabeçalho de um CSV interno sem carregar o arquivo completo."""
    sample = read_sample_text(path, member)
    sep = sniff_delimiter_from_text(sample)
    return sample.splitlines()[0].split(sep)


In [ ]:
def normalize_numeric(series: pd.Series) -> pd.Series:
    """Converte uma série para numérico aceitando vírgula decimal e valores inválidos."""
    if series.dtype == "object":
        series = series.astype(str).str.replace(",", ".", regex=False)
    return pd.to_numeric(series, errors="coerce")


In [ ]:
def iter_zip_csv_chunks(path: Path, member: str, usecols: list[str], chunksize: int = CHUNKSIZE):
    """Itera um CSV compactado em chunks com colunas selecionadas."""
    sample = read_sample_text(path, member)
    sep = sniff_delimiter_from_text(sample)
    with zipfile.ZipFile(path) as zf:
        with zf.open(member) as fh:
            yield from pd.read_csv(
                fh,
                sep=sep,
                encoding="latin1",
                usecols=usecols,
                chunksize=chunksize,
                low_memory=False,
            )


In [ ]:
def available_usecols(path: Path, member: str, requested: list[str]) -> tuple[list[str], list[str]]:
    """Separa colunas solicitadas entre disponíveis e ausentes no arquivo real."""
    header = read_header(path, member)
    available = [col for col in requested if col in header]
    missing = [col for col in requested if col not in header]
    return available, missing


In [ ]:
def prepare_score_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza colunas essenciais de ano, presença e nota antes dos filtros."""
    out = df.copy()
    for col in ["TP_PRESENCA_MT", TARGET_SCORE_COLUMN]:
        if col in out.columns:
            out[col] = normalize_numeric(out[col])
    if "NU_ANO" in out.columns:
        out["NU_ANO"] = normalize_numeric(out["NU_ANO"]).astype("Int64")
    return out


## Descoberta dos arquivos oficiais ou cache publico

Esta etapa confirma quais arquivos estão presentes. Em execução local sem cache, os CSVs internos dos ZIPs oficiais são inventariados. Em 2024, o arquivo de resultados e o arquivo de participantes são separados; isso é documentado explicitamente porque afeta a metodologia. Em Colab, o inventário dos ZIPs é substituído pelo uso das bases derivadas públicas em `outputs/`.


In [ ]:
_cache_results = OUTPUTS / "enem_rj_results_2022_2024.csv.gz"
_cache_profiles = OUTPUTS / "enem_rj_profiles_2022_2024.csv.gz"
_use_cache = _cache_results.exists() and _cache_profiles.exists()

if _use_cache:
    # Modo cache publico/local: os ZIPs brutos do INEP nao sao necessarios.
    # O inventario e pulado; path_by_year fica vazio (nao e usado quando cache existe).
    enem_files = []
    path_by_year = {}
    inventory_df = pd.DataFrame(columns=[
        "ano", "arquivo_zip", "tamanho_mb",
        "csv_microdados", "colunas_microdados",
        "csv_resultados", "colunas_resultados",
        "csv_participantes", "colunas_participantes",
    ])
    print("Cache pré-processado encontrado — inventário de ZIPs pulado.")
else:
    enem_files = ensure_enem_files_available()
    path_by_year = {infer_year_from_name(path): path for path in enem_files}

    inventory_rows = []
    for year, path in sorted(path_by_year.items()):
        row = {"ano": year, "arquivo_zip": path.name, "tamanho_mb": round(path.stat().st_size / (1024 ** 2), 2)}
        for kind in ["microdados", "resultados", "participantes"]:
            try:
                member = select_csv_member(path, kind)
                row[f"csv_{kind}"] = member
                row[f"colunas_{kind}"] = len(read_header(path, member))
            except FileNotFoundError:
                row[f"csv_{kind}"] = None
                row[f"colunas_{kind}"] = None
        inventory_rows.append(row)

    inventory_df = pd.DataFrame(inventory_rows)
    display(inventory_df)
    inventory_df.to_csv(ARTIFACTS / "source_file_inventory.csv", index=False)

    assert set(path_by_year) == EXPECTED_YEARS
    assert "RESULTADOS_2024" in inventory_df.loc[inventory_df["ano"] == 2024, "csv_resultados"].iloc[0]
    assert "PARTICIPANTES_2024" in inventory_df.loc[inventory_df["ano"] == 2024, "csv_participantes"].iloc[0]


## Carregamento dos dados analíticos

Esta etapa carrega apenas as colunas necessárias. No fluxo local a leitura dos ZIPs oficiais ocorre em chunks, reduzindo uso de memória e registrando linhas lidas, linhas mantidas e colunas ausentes por ano. No Colab, as mesmas bases analíticas já derivadas dos microdados oficiais são lidas a partir dos `CSV.gz` públicos.

Em 2024, o INEP separa participantes e resultados. O notebook respeita essa diferença e não tenta reconstruir vínculo individual entre as duas bases.


In [ ]:
def read_results_for_year(year: int) -> tuple[pd.DataFrame, dict]:
    """Carrega resultados de um ano aplicando recorte RJ, presença em Matemática e nota válida."""
    path = path_by_year[year]
    member = select_csv_member(path, "resultados")
    requested = REQUESTED_2024_RESULT_COLUMNS if year == 2024 else REQUESTED_2022_2023_COLUMNS
    usecols, missing = available_usecols(path, member, requested)
    required = {"NU_ANO", "SG_UF_PROVA", "TP_PRESENCA_MT", TARGET_SCORE_COLUMN}
    missing_required = sorted(required - set(usecols))
    if missing_required:
        raise ValueError(f"{year}: colunas obrigatorias ausentes em {member}: {missing_required}")

    chunks = []
    rows_seen = rows_kept = 0
    for chunk in iter_zip_csv_chunks(path, member, usecols):
        rows_seen += len(chunk)
        chunk = prepare_score_columns(chunk)
        filtered = chunk[
            (chunk["SG_UF_PROVA"].astype(str).str.upper() == RECORTE_UF)
            & (chunk["TP_PRESENCA_MT"] == 1)
            & (chunk[TARGET_SCORE_COLUMN].notna())
        ].copy()
        rows_kept += len(filtered)
        if not filtered.empty:
            chunks.append(filtered)
    df_year = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=usecols)
    meta = {
        "year": year,
        "zip_file": path.name,
        "csv_member": member,
        "rows_seen": int(rows_seen),
        "rows_kept_score_valid_rj": int(rows_kept),
        "missing_requested_columns": missing,
    }
    return df_year, meta


In [ ]:
def read_profile_for_year(year: int) -> tuple[pd.DataFrame, dict]:
    """Carrega dados de perfil de um ano aplicando apenas o recorte RJ de prova."""
    path = path_by_year[year]
    member = select_csv_member(path, "participantes")
    requested = REQUESTED_2024_PARTICIPANT_COLUMNS if year == 2024 else REQUESTED_2022_2023_COLUMNS
    usecols, missing = available_usecols(path, member, requested)
    required = {"NU_ANO", "SG_UF_PROVA", "CO_MUNICIPIO_PROVA"}
    missing_required = sorted(required - set(usecols))
    if missing_required:
        raise ValueError(f"{year}: colunas obrigatorias de perfil ausentes em {member}: {missing_required}")

    chunks = []
    rows_seen = rows_kept = 0
    for chunk in iter_zip_csv_chunks(path, member, usecols):
        rows_seen += len(chunk)
        if "NU_ANO" in chunk.columns:
            chunk["NU_ANO"] = normalize_numeric(chunk["NU_ANO"]).astype("Int64")
        filtered = chunk[chunk["SG_UF_PROVA"].astype(str).str.upper() == RECORTE_UF].copy()
        rows_kept += len(filtered)
        if not filtered.empty:
            chunks.append(filtered)
    df_year = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=usecols)
    meta = {
        "year": year,
        "zip_file": path.name,
        "csv_member": member,
        "rows_seen": int(rows_seen),
        "rows_kept_profile_rj": int(rows_kept),
        "missing_requested_columns": missing,
    }
    return df_year, meta


In [ ]:
_cache_results = OUTPUTS / "enem_rj_results_2022_2024.csv.gz"
_cache_profiles = OUTPUTS / "enem_rj_profiles_2022_2024.csv.gz"

if _cache_results.exists() and _cache_profiles.exists():
    print("Carregando dados do cache (CSV.gz)...")
    try:
        df_results = pd.read_csv(_cache_results, low_memory=False)
        df_profiles = pd.read_csv(_cache_profiles, low_memory=False)
    except Exception as exc:
        raise RuntimeError(
            "Falha ao ler os caches em outputs/. Remova os arquivos parciais e execute novamente."
        ) from exc
    for _frame in [df_results, df_profiles]:
        _frame["NU_ANO"] = normalize_numeric(_frame["NU_ANO"]).astype(int)
        _frame["CO_MUNICIPIO_PROVA"] = normalize_numeric(_frame["CO_MUNICIPIO_PROVA"]).astype("Int64")
    result_metadata = [
        {
            "year": y,
            "rows_seen": -1,
            "rows_kept_score_valid_rj": int((df_results["NU_ANO"] == y).sum()),
            "csv_member": "cache/enem_rj_results_2022_2024.csv.gz",
        }
        for y in sorted(EXPECTED_YEARS)
    ]
    profile_metadata = [
        {
            "year": y,
            "rows_kept_profile_rj": int((df_profiles["NU_ANO"] == y).sum()),
            "csv_member": "cache/enem_rj_profiles_2022_2024.csv.gz",
        }
        for y in sorted(EXPECTED_YEARS)
    ]
    print(f"  df_results: {df_results.shape}")
    print(f"  df_profiles: {df_profiles.shape}")
else:
    results_frames, result_metadata = [], []
    profile_frames, profile_metadata = [], []

    for year in sorted(EXPECTED_YEARS):
        print(f"Lendo resultados {year}...")
        df_result_year, meta_result = read_results_for_year(year)
        results_frames.append(df_result_year)
        result_metadata.append(meta_result)
        print(f"  mantidos resultados RJ validos: {meta_result['rows_kept_score_valid_rj']:,}")

        print(f"Lendo perfil {year}...")
        df_profile_year, meta_profile = read_profile_for_year(year)
        profile_frames.append(df_profile_year)
        profile_metadata.append(meta_profile)
        print(f"  mantidos perfis RJ: {meta_profile['rows_kept_profile_rj']:,}")

    df_results = pd.concat(results_frames, ignore_index=True)
    df_profiles = pd.concat(profile_frames, ignore_index=True)

    for frame in [df_results, df_profiles]:
        frame["NU_ANO"] = normalize_numeric(frame["NU_ANO"]).astype(int)
        frame["CO_MUNICIPIO_PROVA"] = normalize_numeric(frame["CO_MUNICIPIO_PROVA"]).astype("Int64")

    df_results.to_csv(OUTPUTS / "enem_rj_results_2022_2024.csv.gz", index=False, compression="gzip")
    df_profiles.to_csv(OUTPUTS / "enem_rj_profiles_2022_2024.csv.gz", index=False, compression="gzip")

pd.DataFrame(result_metadata).to_csv(ARTIFACTS / "result_source_metadata.csv", index=False)
pd.DataFrame(profile_metadata).to_csv(ARTIFACTS / "profile_source_metadata.csv", index=False)

print("Resultados elegiveis:", df_results.shape)
print("Perfis RJ:", df_profiles.shape)
display(pd.DataFrame(result_metadata))
display(pd.DataFrame(profile_metadata))

assert set(df_results["NU_ANO"].unique()) == EXPECTED_YEARS
assert set(df_profiles["NU_ANO"].unique()) == EXPECTED_YEARS


## Pré-processamento

O pré-processamento é separado em duas camadas. A primeira camada ocorre antes da modelagem e define o recorte analítico: ano, UF de prova, presença em Matemática, nota válida, split e target. A segunda camada ocorre dentro de `Pipeline` e `ColumnTransformer`: imputação, codificação categórica e escala numérica.

Essa separação reduz vazamento. O target e os filtros são documentados; transformações aprendidas, como imputação e one-hot encoding, são ajustadas apenas no treino de cada experimento.


In [ ]:
preprocessing_decisions = pd.DataFrame([
    {
        "etapa": "Filtro geográfico",
        "colunas": "SG_UF_PROVA",
        "transformacao": "Manter apenas participantes com prova no RJ.",
        "motivo": "Definir recorte territorial do MVP.",
        "impacto_esperado": "Comparabilidade regional e narrativa focada no RJ.",
        "onde_ocorre": "Leitura em chunks.",
        "mitigacao_leakage": "SG_UF_PROVA é removida quando constante ou proibida no experimento.",
    },
    {
        "etapa": "Elegibilidade em Matemática",
        "colunas": "TP_PRESENCA_MT, NU_NOTA_MT",
        "transformacao": "Manter presença em Matemática e nota válida.",
        "motivo": "O target depende da nota de Matemática observada.",
        "impacto_esperado": "Remove ausências sem resultado avaliável.",
        "onde_ocorre": "Leitura em chunks de resultados.",
        "mitigacao_leakage": "Presença e notas não entram nas features.",
    },
    {
        "etapa": "Target por P75",
        "colunas": "NU_NOTA_MT",
        "transformacao": "Criar target binário com P75 calculado apenas no treino.",
        "motivo": "Definir alto desempenho de forma seletiva e interpretável.",
        "impacto_esperado": "Classe positiva menor, exigindo F1-score como métrica principal.",
        "onde_ocorre": "Após split no Modelo A; apenas treino temporal no Modelo B.",
        "mitigacao_leakage": "Teste e 2024 não influenciam o limiar.",
    },
    {
        "etapa": "Imputação categórica",
        "colunas": "Features categóricas permitidas.",
        "transformacao": "SimpleImputer(strategy='most_frequent').",
        "motivo": "Modelos do scikit-learn não aceitam ausências brutas.",
        "impacto_esperado": "Mantém registros sem imputação manual fora do treino.",
        "onde_ocorre": "Pipeline/ColumnTransformer.",
        "mitigacao_leakage": "Imputer é ajustado apenas no treino.",
    },
    {
        "etapa": "Codificação categórica",
        "colunas": "Features categóricas permitidas.",
        "transformacao": "OneHotEncoder(handle_unknown='ignore').",
        "motivo": "Transformar categorias em matriz numérica para os modelos.",
        "impacto_esperado": "Permite categorias novas no teste sem erro.",
        "onde_ocorre": "Pipeline/ColumnTransformer.",
        "mitigacao_leakage": "Categorias são aprendidas apenas no treino.",
    },
    {
        "etapa": "Escala numérica",
        "colunas": "Agregados municipais do Modelo B.",
        "transformacao": "SimpleImputer(strategy='median') + StandardScaler().",
        "motivo": "Apoiar modelo linear e manter tratamento reproduzível.",
        "impacto_esperado": "Coeficientes e otimização mais estáveis para features numéricas.",
        "onde_ocorre": "Pipeline/ColumnTransformer.",
        "mitigacao_leakage": "Mediana e escala são ajustadas apenas no treino.",
    },
])

display(preprocessing_decisions)
preprocessing_decisions.to_csv(ARTIFACTS / "preprocessing_decisions.csv", index=False)
assert preprocessing_decisions["onde_ocorre"].str.contains("Pipeline/ColumnTransformer", regex=False).any()


## Apresentação e qualidade dos dados

Esta etapa documenta quantidade de registros, atributos, tipos, valores ausentes e distribuição da nota de Matemática. Ela atende à exigência de apresentar o dataset antes da modelagem.


In [ ]:
dataset_summary = (
    df_results.groupby("NU_ANO")
    .agg(
        participantes_validos=(TARGET_SCORE_COLUMN, "size"),
        nota_mt_media=(TARGET_SCORE_COLUMN, "mean"),
        nota_mt_mediana=(TARGET_SCORE_COLUMN, "median"),
        nota_mt_p75=(TARGET_SCORE_COLUMN, lambda s: s.quantile(0.75)),
    )
    .reset_index()
)
profile_summary = df_profiles.groupby("NU_ANO").size().rename("participantes_perfil_rj").reset_index()

schema_summary = pd.DataFrame({
    "coluna": df_results.columns,
    "tipo": [str(dtype) for dtype in df_results.dtypes],
    "valores_na": df_results.isna().sum().values,
    "percentual_na": (df_results.isna().mean().values * 100).round(2),
    "valores_unicos": [df_results[col].nunique(dropna=True) for col in df_results.columns],
}).sort_values(["percentual_na", "coluna"], ascending=[False, True])

score_describe = (
    df_results.groupby("NU_ANO")[TARGET_SCORE_COLUMN]
    .describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    .reset_index()
)

display(df_results.head())
display(dataset_summary)
display(profile_summary)
display(schema_summary.head(25))
display(score_describe)

dataset_summary.to_csv(ARTIFACTS / "dataset_summary_by_year.csv", index=False)
profile_summary.to_csv(ARTIFACTS / "profile_summary_by_year.csv", index=False)
schema_summary.to_csv(ARTIFACTS / "data_quality_schema_missing.csv", index=False)
score_describe.to_csv(ARTIFACTS / "score_descriptive_stats_by_year.csv", index=False)

assert df_results[TARGET_SCORE_COLUMN].between(0, 1000).all()


## Dicionário do Questionário Socioeconômico (Q001–Q025)

O ENEM inclui um questionário socioeconômico respondido no ato de inscrição. As 25 variáveis (Q001–Q025) cobrem composição familiar, bens domésticos, escolaridade dos pais e trajetória escolar. Em 2022 e 2023 o questionário tem Q001–Q025; em **2024 vai apenas até Q023 e foi renumerado** — em particular, `Q006` deixa de ser a pergunta de renda (17 faixas A–Q) e passa a ser uma pergunta **binária** (só A/B), como demonstra a EDA temporal adiante (diferença de schema que impede agregados comparáveis diretos para as duas últimas variáveis).

Sem esse dicionário, códigos como `Q006_A` ou `Q002_E` são opacos. A tabela abaixo cobre as **variáveis com maior relevância neste MVP**:

| Código | Tema | Escala resumida (A → máximo) |
|---|---|---|
| **Q001** | Com quem mora | A=pai e mãe · B=só mãe · C=só pai · D=avós · E=outros familiares · F=cônjuge/filhos · G=sozinho · H=pensionato/alojamento |
| **Q002** | Escolaridade da **mãe** | A=nunca estudou · B=até 5º ano EF · C=até 9º ano EF · D=EM incompleto · E=EM completo · F=Sup incompleto · G=Sup completo · H=pós-graduação |
| **Q003** | Escolaridade do **pai** | *mesma escala de Q002* |
| **Q005** | Pessoas na casa | A=1 · B=2 · C=3 · D=4 · E=5 · F=6 · G=7 ou mais |
| **Q006** | **Renda mensal familiar** ⭐ | A=nenhuma renda · B=até R$1.320 · C–F=R$1.320–R$3.960 · G–J=R$3.960–R$7.920 · K–O=R$7.920–R$15.840 · P–Q=acima de R$15.840 (17 faixas no total, ajustadas por edição) |
| **Q007** | Empregado(a) doméstico(a) | A=não · B=1 · C=2 · D=3 ou mais |
| **Q008** | Banheiros | A=nenhum · B=1 · C=2 · D=3 · E=4 ou mais |
| **Q009** | Automóveis | A=nenhum · B=1 · C=2 · D=3 · E=4 ou mais |
| **Q020** | Computador em casa | A=não · B=1 · C=2 · D=3 ou mais |
| **Q021** | Acesso à internet em casa | A=não · B=sim |
| **Q024** | Turno do Ensino Médio | A=matutino · B=vespertino · C=noturno · D=integral |
| **Q025** | Atividade remunerada | A=não · B=sim, até 20h/sem · C=20–40h/sem · D=mais de 40h/sem |

> **⭐ Q006 é a variável mais relevante do Modelo A.** Seus coeficientes log-odds confirmam o gradiente de desigualdade: faixas baixas (A–C) têm coeficiente negativo (−0,94 a −0,53) e faixas altas (O–Q) têm coeficiente positivo (+0,41 a +0,68). Ver Seção "Interpretabilidade" para detalhes.
>
> O dicionário completo (incluindo todos os valores de cada questão) está na [documentação oficial do INEP](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem), arquivo `Dicionário_Microdados_ENEM_20XX.xlsx`.

## Análise exploratória inicial

A EDA responde três perguntas antes da modelagem: como a nota de Matemática varia no tempo, como o alto desempenho se distribui e quais grupos socioeducacionais apresentam diferenças descritivas.

Essas tabelas não provam causalidade. Elas orientam a narrativa e ajudam a interpretar os resultados dos modelos.


In [ ]:
def grouped_score_table(data: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Resume nota e proporção de alto desempenho por ano e grupo categórico."""
    if group_col not in data.columns:
        return pd.DataFrame()
    table = (
        data.groupby(["NU_ANO", group_col], dropna=False)
        .agg(
            participantes=(TARGET_SCORE_COLUMN, "size"),
            nota_mt_media=(TARGET_SCORE_COLUMN, "mean"),
            proporcao_alto=(TARGET_COLUMN, "mean"),
        )
        .reset_index()
        .sort_values(["NU_ANO", "participantes"], ascending=[True, False])
    )
    return table


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for year in sorted(EXPECTED_YEARS):
    scores = df_results.loc[df_results["NU_ANO"] == year, TARGET_SCORE_COLUMN]
    ax.hist(scores, bins=35, alpha=0.35, density=True, label=str(year), color=YEAR_COLORS[year])
ax.set_title("Distribuição da nota de Matemática no RJ por ano")
ax.set_xlabel("Nota de Matemática")
ax.set_ylabel("Densidade")
ax.legend(title="Ano")
fig.tight_layout()
fig.savefig(ARTIFACTS / "nota_mt_distribuicao_por_ano.png", dpi=160)
plt.show()

eda_a_base = df_results[df_results["NU_ANO"].isin(TRAIN_YEARS)].copy()
eda_a_base[TARGET_COLUMN] = (eda_a_base[TARGET_SCORE_COLUMN] >= eda_a_base[TARGET_SCORE_COLUMN].quantile(TARGET_QUANTILE)).astype(int)


school_table = grouped_score_table(eda_a_base, "TP_ESCOLA")
race_table = grouped_score_table(eda_a_base, "TP_COR_RACA")
income_table = grouped_score_table(eda_a_base, "Q006")

display(school_table.head(30))
display(race_table.head(30))
display(income_table.head(30))

school_table.to_csv(ARTIFACTS / "eda_model_a_by_school_type.csv", index=False)
race_table.to_csv(ARTIFACTS / "eda_model_a_by_race_color.csv", index=False)
income_table.to_csv(ARTIFACTS / "eda_model_a_by_income_proxy_q006.csv", index=False)

if not school_table.empty:
    pivot_school = school_table.pivot(index="TP_ESCOLA", columns="NU_ANO", values="nota_mt_media")
    ax = pivot_school.plot(kind="bar", figsize=(10, 5), color=[YEAR_COLORS[year] for year in pivot_school.columns])
    ax.set_title("Nota média de Matemática por tipo de escola — 2022-2023")
    ax.set_xlabel("TP_ESCOLA")
    ax.set_ylabel("Nota média de Matemática")
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(ARTIFACTS / "model_a_nota_mt_media_por_tipo_escola.png", dpi=160)
    plt.show()


## Diferenças Entre os Dois Experimentos

**Modelo A:** utiliza atributos individuais, incluindo perfil, trajetória escolar e questionário socioeconômico. Seu objetivo é interpretativo e explicativo: responder à pergunta socioeducacional principal.

**Modelo B:** utiliza atributos comparáveis ao longo do tempo e agregados municipais. Seu objetivo é testar robustez temporal: avaliar se parte do padrão aprendido em 2022-2023 se mantém em 2024.

**Os dois modelos respondem perguntas diferentes e não devem ser interpretados como uma competição direta entre algoritmos.** Métricas de A e B são analisadas dentro de cada experimento, não como ranking único.


## Justificativa do target por P75

O percentil 75 foi escolhido porque representa um critério seletivo e interpretável de alto desempenho: o grupo no quarto superior da distribuição de Matemática. Essa definição é mais alinhada ao objetivo do MVP do que prever a média ou a mediana, pois foca em identificar desempenho elevado.

O limiar é calculado exclusivamente com dados de treino em cada experimento. Calcular o P75 usando teste ou 2024 permitiria que a distribuição dos dados de avaliação influenciasse a própria definição da variável-alvo, criando vazamento de dados e superestimando a validade da avaliação.

O princípio aplicado é: **o conjunto de avaliação nunca participa da definição do target, da otimização de hiperparâmetros ou do ajuste de transformações.**


# Experimento A — Modelo Socioeducacional Individual

Este é o experimento principal para a pergunta socioeducacional. Ele usa 2022-2023, onde perfil individual, questionário e resultados estão no mesmo registro. A avaliação é feita por divisão treino/teste estratificada dentro desses anos.


## Modelo A: preparação, split e target

Primeiro separam-se treino e teste. Depois o limiar P75 é calculado apenas no treino. Isso evita que a distribuição do teste influencie a construção do target.


In [ ]:
model_a_base = df_results[df_results["NU_ANO"].isin(TRAIN_YEARS)].copy()
model_a_base = model_a_base.dropna(subset=[TARGET_SCORE_COLUMN]).copy()

model_a_train_df, model_a_test_df = train_test_split(
    model_a_base,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=model_a_base["NU_ANO"].astype(str),
)
model_a_threshold = model_a_train_df[TARGET_SCORE_COLUMN].quantile(TARGET_QUANTILE)
for frame in [model_a_train_df, model_a_test_df]:
    frame[TARGET_COLUMN] = (frame[TARGET_SCORE_COLUMN] >= model_a_threshold).astype(int)

model_a_feature_candidates = PROFILE_COLUMNS + SCHOOL_COLUMNS + PROVA_COLUMNS + QUESTIONARIO_COLUMNS_2022_2023
model_a_forbidden = set(PRESENCE_COLUMNS + SCORE_COLUMNS + REDACAO_COMP_COLUMNS + ["NU_INSCRICAO", "NU_ANO", TARGET_COLUMN])
model_a_features = [col for col in model_a_feature_candidates if col in model_a_train_df.columns and col not in model_a_forbidden]

constant_features_a = sorted(col for col in model_a_features if model_a_train_df[col].nunique(dropna=False) <= 1)
model_a_features = [col for col in model_a_features if col not in constant_features_a]
print(f"Features constantes removidas (recorte {RECORTE_UF}): {constant_features_a}")

X_a_train_full = model_a_train_df[model_a_features].copy()
y_a_train_full = model_a_train_df[TARGET_COLUMN].copy()
X_a_test = model_a_test_df[model_a_features].copy()
y_a_test = model_a_test_df[TARGET_COLUMN].copy()

model_a_split_summary = pd.DataFrame({
    "particao": ["treino_a", "teste_a"],
    "linhas": [len(X_a_train_full), len(X_a_test)],
    "proporcao_target_1": [y_a_train_full.mean(), y_a_test.mean()],
})
display(Markdown(f"Limiar P75 do Modelo A calculado apenas no treino: **{model_a_threshold:.2f}**."))
display(model_a_split_summary)

assert len(X_a_train_full) > 0 and len(X_a_test) > 0
assert set(y_a_train_full.unique()).issubset({0, 1})
assert TARGET_SCORE_COLUMN not in model_a_features
assert not any(col.startswith(LEAKAGE_PREFIXES) for col in model_a_features)
assert all(model_a_train_df[col].nunique(dropna=False) > 1 for col in model_a_features)


## Modelo A: auditoria das features

Esta auditoria mostra quais variáveis individuais foram usadas e quais foram removidas por vazamento, contexto ou indisponibilidade. Ela torna a preparação dos dados verificável.


In [ ]:
def build_feature_audit(all_columns: list[str], used_features: list[str], forbidden_columns: set[str], unavailable_columns: set[str] | None = None) -> pd.DataFrame:
    """Classifica colunas como usadas ou removidas e registra a justificativa metodológica."""
    unavailable_columns = unavailable_columns or set()
    rows = []
    for col in sorted(all_columns):
        if col in used_features:
            status, reason = "usada", "Feature permitida no desenho do experimento."
        elif col in forbidden_columns or col.startswith(LEAKAGE_PREFIXES):
            status, reason = "removida", "Vazamento, target, presença, nota, resposta, gabarito, prova ou identificador."
        elif col in unavailable_columns:
            status, reason = "removida", "Indisponível ou incompatível para este experimento."
        else:
            status, reason = "removida", "Fora da lista de features definida para o experimento."
        rows.append({"coluna": col, "status": status, "motivo": reason})
    return pd.DataFrame(rows)


In [ ]:
feature_audit_model_a = build_feature_audit(list(model_a_train_df.columns), model_a_features, model_a_forbidden)
display(feature_audit_model_a)
feature_audit_model_a.to_csv(ARTIFACTS / "feature_audit_model_a.csv", index=False)
assert set(model_a_features) == set(feature_audit_model_a.loc[feature_audit_model_a["status"] == "usada", "coluna"])


> **Nota sobre features constantes no Modelo A:** no recorte `SG_UF_PROVA == "RJ"`, as variáveis `SG_UF_PROVA` e `CO_UF_PROVA` tornam-se constantes (100% das observações com o mesmo valor) e não carregam nenhuma informação preditiva. Conforme a regra do projeto (`FEATURE_LEAKAGE_RULES.md`: *"não use SG_UF_PROVA como feature se ela se tornar constante"*) e a documentação de pré-processamento, a célula de preparação **detecta e remove programaticamente qualquer feature constante** antes do treino — a lista removida é impressa na saída da célula de split acima. Na auditoria de features, essas colunas aparecem com o status "removida" e motivo "fora da lista de features".

## Funções de modelagem e avaliação

As mesmas funções são usadas nos dois experimentos para manter comparação justa dentro de cada experimento: mesmo tipo de pipeline, mesmas métricas, mesma lógica de baseline e otimização.


In [ ]:
def clean_feature_matrix(X: pd.DataFrame, categorical_features: list[str], numeric_features: list[str]) -> pd.DataFrame:
    """Padroniza tipos antes de enviar features ao Pipeline do scikit-learn."""
    X_clean = X.copy()
    for col in categorical_features:
        if col in X_clean.columns:
            X_clean[col] = X_clean[col].astype("object").where(X_clean[col].notna(), np.nan)
    for col in numeric_features:
        if col in X_clean.columns:
            X_clean[col] = pd.to_numeric(X_clean[col], errors="coerce")
    return X_clean


In [ ]:
def make_preprocessor(categorical_features: list[str], numeric_features: list[str]) -> ColumnTransformer:
    """Cria um ColumnTransformer com imputação, one-hot encoding e escala quando aplicável."""
    transformers = []
    if categorical_features:
        transformers.append((
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ))
    if numeric_features:
        transformers.append((
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ))
    return ColumnTransformer(transformers)


In [ ]:
def make_pipeline(model, categorical_features: list[str], numeric_features: list[str]) -> Pipeline:
    """Combina pré-processamento e estimador em um Pipeline reproduzível."""
    return Pipeline([
        ("preprocessor", make_preprocessor(categorical_features, numeric_features)),
        ("model", model),
    ])


In [ ]:
def safe_roc_auc(y_true: pd.Series, y_score: np.ndarray) -> float:
    """Calcula ROC-AUC quando as duas classes estão presentes no conjunto avaliado."""
    return float(roc_auc_score(y_true, y_score)) if pd.Series(y_true).nunique() == 2 else np.nan


In [ ]:
def get_score(estimator, X_eval: pd.DataFrame) -> np.ndarray:
    """Obtém escore contínuo ou probabilidade para métricas baseadas em ranking."""
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X_eval)[:, 1]
    if hasattr(estimator, "decision_function"):
        return estimator.decision_function(X_eval)
    return estimator.predict(X_eval)


In [ ]:
def evaluate_classifier(name: str, estimator, X_eval: pd.DataFrame, y_eval: pd.Series, split_name: str, training_seconds: float | None = None) -> dict:
    """Calcula métricas com uma única passagem de inferência sempre que possível."""
    if hasattr(estimator, "predict_proba"):
        y_score = estimator.predict_proba(X_eval)[:, 1]
        y_pred = (y_score >= 0.5).astype(int)
    elif hasattr(estimator, "decision_function"):
        y_score = estimator.decision_function(X_eval)
        y_pred = (y_score >= 0.0).astype(int)
    else:
        y_pred = estimator.predict(X_eval)
        y_score = y_pred
    return {
        "modelo": name,
        "split": split_name,
        "accuracy": float(accuracy_score(y_eval, y_pred)),
        "precision": float(precision_score(y_eval, y_pred, zero_division=0)),
        "recall": float(recall_score(y_eval, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_eval, y_pred, zero_division=0)),
        "roc_auc": safe_roc_auc(y_eval, y_score),
        "confusion_matrix": confusion_matrix(y_eval, y_pred, labels=[0, 1]).tolist(),
        "training_seconds": None if training_seconds is None else float(training_seconds),
    }


In [ ]:
def sample_training_data(X: pd.DataFrame, y: pd.Series, max_rows: int = MAX_TRAIN_ROWS_FOR_MODELING) -> tuple[pd.DataFrame, pd.Series]:
    """Amostra antes da limpeza, preservando target e ano quando o ano auxiliar está presente."""
    if len(X) <= max_rows:
        return X.copy(), y.copy()
    temp = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True).rename(TARGET_COLUMN)], axis=1)
    strata = [TARGET_COLUMN]
    if "__sampling_year" in temp.columns:
        strata.append("__sampling_year")
    sampled = temp.groupby(strata, group_keys=False).sample(frac=max_rows / len(temp), random_state=RANDOM_STATE)
    return sampled[X.columns].copy(), sampled[TARGET_COLUMN].copy()


In [ ]:
def run_model_suite(
    experiment_label: str,
    X_train_full: pd.DataFrame,
    y_train_full: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    categorical_features: list[str],
    numeric_features: list[str],
    test_split_name: str,
    selection_years: pd.Series,
    selection_mode: str = "stratified",
) -> tuple[pd.DataFrame, pd.DataFrame, dict, str]:
    """Seleciona no treino, ajusta os modelos finais e só então consulta o teste."""
    selection_source = X_train_full.copy()
    selection_source["__sampling_year"] = selection_years.reset_index(drop=True).to_numpy()
    X_selection, y_selection = sample_training_data(selection_source, y_train_full)
    selection_year_sample = X_selection.pop("__sampling_year").astype(int).to_numpy()
    X_selection = clean_feature_matrix(X_selection, categorical_features, numeric_features)
    X_test_clean = clean_feature_matrix(X_test, categorical_features, numeric_features)

    if selection_mode == "temporal":
        selection_train_idx = np.flatnonzero(selection_year_sample == min(TRAIN_YEARS))
        selection_valid_idx = np.flatnonzero(selection_year_sample == max(TRAIN_YEARS))
        cv_splits = [(selection_train_idx, selection_valid_idx)]
        tie_margin = MODEL_B_TIE_MARGIN
    else:
        splitter = StratifiedKFold(n_splits=MODEL_SELECTION_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        cv_splits = list(splitter.split(X_selection, y_selection))
        tie_margin = MODEL_A_TIE_MARGIN

    rf_pipeline = make_pipeline(
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=MODEL_PARALLEL_JOBS,
            class_weight="balanced_subsample",
        ),
        categorical_features,
        numeric_features,
    )
    param_grid = [
        {"model__n_estimators": [RF_SEARCH_N_ESTIMATORS], "model__max_depth": [16], "model__min_samples_leaf": [3]},
        {"model__n_estimators": [RF_SEARCH_N_ESTIMATORS], "model__max_depth": [12], "model__min_samples_leaf": [8]},
        {"model__n_estimators": [RF_SEARCH_N_ESTIMATORS], "model__max_depth": [20], "model__min_samples_leaf": [8]},
    ]
    grid_search = GridSearchCV(
        rf_pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=cv_splits,
        n_jobs=1,
        pre_dispatch=MODEL_SELECTION_PRE_DISPATCH,
        refit=False,
        return_train_score=False,
    )
    search_start = time.perf_counter()
    grid_search.fit(X_selection, y_selection)
    search_seconds = time.perf_counter() - search_start

    lr_selection = make_pipeline(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
        categorical_features,
        numeric_features,
    )
    lr_cv_scores = cross_val_score(
        lr_selection,
        X_selection,
        y_selection,
        cv=cv_splits,
        scoring="f1",
        n_jobs=1,
        pre_dispatch=MODEL_SELECTION_PRE_DISPATCH,
    )
    lr_cv_mean = float(lr_cv_scores.mean())
    rf_cv_mean = float(grid_search.best_score_)
    best_rf_params = dict(grid_search.best_params_)
    simple_rf_params = {
        "model__max_depth": 16,
        "model__min_samples_leaf": 3,
        "model__n_estimators": RF_SEARCH_N_ESTIMATORS,
    }
    best_rf_name = "RandomForestClassifier" if best_rf_params == simple_rf_params else "RandomForestClassifier_Optimized"
    best_model_name = "LogisticRegression" if lr_cv_mean >= rf_cv_mean - tie_margin else best_rf_name

    selection_rows = [{
        "modelo": "LogisticRegression",
        "parametros": {"max_iter": 1000, "class_weight": "balanced"},
        "f1_validacao_media": lr_cv_mean,
        "f1_validacao_desvio": float(lr_cv_scores.std()),
        "selecionado": best_model_name == "LogisticRegression",
    }]
    for idx, params in enumerate(grid_search.cv_results_["params"]):
        candidate_name = "RandomForestClassifier" if params == simple_rf_params else f"RandomForest_candidato_{idx + 1}"
        if params == best_rf_params:
            candidate_name = best_rf_name
        selection_rows.append({
            "modelo": candidate_name,
            "parametros": params,
            "f1_validacao_media": float(grid_search.cv_results_["mean_test_score"][idx]),
            "f1_validacao_desvio": float(grid_search.cv_results_["std_test_score"][idx]),
            "selecionado": candidate_name == best_model_name,
        })
    selection_results = pd.DataFrame(selection_rows).sort_values("f1_validacao_media", ascending=False).reset_index(drop=True)

    model_specs = {
        "DummyClassifier": (DummyClassifier(strategy="most_frequent"), X_selection, y_selection),
    }
    if best_model_name == "LogisticRegression":
        X_train_final = clean_feature_matrix(X_train_full, categorical_features, numeric_features)
        y_train_final = y_train_full.reset_index(drop=True)
        final_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
    else:
        X_train_final = X_selection
        y_train_final = y_selection
        final_model = RandomForestClassifier(
            n_estimators=RF_FINAL_N_ESTIMATORS,
            max_depth=best_rf_params["model__max_depth"],
            min_samples_leaf=best_rf_params["model__min_samples_leaf"],
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=MODEL_PARALLEL_JOBS,
        )
    model_specs[best_model_name] = (final_model, X_train_final, y_train_final)
    selected_rf_refit_performed = best_model_name.startswith("RandomForest")

    results = []
    trained_models = {}
    for name, (model, X_fit, y_fit) in model_specs.items():
        estimator = make_pipeline(model, categorical_features, numeric_features)
        start = time.perf_counter()
        estimator.fit(X_fit, y_fit)
        elapsed = time.perf_counter() - start
        trained_models[name] = estimator
        results.append(evaluate_classifier(name, estimator, X_fit, y_fit, "treino_modelagem", elapsed))
        results.append(evaluate_classifier(name, estimator, X_test_clean, y_test, test_split_name, elapsed))

    results_long = pd.DataFrame(results)
    test_results = (
        results_long[results_long["split"] == test_split_name]
        .drop(columns=["split"])
        .assign(selecionado_no_treino=lambda frame: frame["modelo"].eq(best_model_name))
        .sort_values(["selecionado_no_treino", "f1_score"], ascending=[False, False])
        .reset_index(drop=True)
    )
    trained_models["grid_search_best_params"] = best_rf_params
    trained_models["selection_results"] = selection_results
    trained_models["selection_search_seconds"] = search_seconds
    trained_models["selected_rf_refit_performed"] = selected_rf_refit_performed
    trained_models["X_train_modeling_rows"] = len(X_selection)
    trained_models["X_train_logistic_rows"] = len(X_train_final) if best_model_name == "LogisticRegression" else 0
    trained_models["final_model_params"] = (
        {"max_iter": 1000, "class_weight": "balanced"}
        if best_model_name == "LogisticRegression"
        else {
            "n_estimators": RF_FINAL_N_ESTIMATORS,
            "max_depth": best_rf_params["model__max_depth"],
            "min_samples_leaf": best_rf_params["model__min_samples_leaf"],
        }
    )

    print(f"{experiment_label}: seleção em {len(X_selection):,} linhas; modelo final em {len(X_train_final):,}.")
    print(f"Modelo bloqueado pela validação do treino: {best_model_name}")
    return test_results, results_long, trained_models, best_model_name


## Modelo A: treinamento, otimização e avaliação

O Modelo A testa H1 e H3 na avaliação convencional 2022-2023. A escolha entre `LogisticRegression` e as configurações da Random Forest é feita por validação cruzada apenas no treino. A busca usa 80 árvores para comparar as configurações da RF; se uma RF for selecionada, o ajuste final usa 120 árvores. Depois da escolha, somente o baseline e o modelo bloqueado são avaliados no teste.


In [ ]:
model_a_categorical_features = model_a_features
model_a_numeric_features: list[str] = []

model_a_suite_start = time.perf_counter()
model_a_results_df, model_a_results_long_df, model_a_trained, model_a_best_name = run_model_suite(
    "Modelo A",
    X_a_train_full,
    y_a_train_full,
    X_a_test,
    y_a_test,
    model_a_categorical_features,
    model_a_numeric_features,
    "teste_a_2022_2023",
    model_a_train_df["NU_ANO"],
    selection_mode="stratified",
)
model_a_suite_seconds = time.perf_counter() - model_a_suite_start

display(Markdown("**Seleção feita apenas no treino:**"))
display(model_a_trained["selection_results"])
model_a_trained["selection_results"].to_csv(ARTIFACTS / "model_a_training_selection.csv", index=False)
display(model_a_results_df.drop(columns=["confusion_matrix"]))
model_a_baseline_f1 = float(model_a_results_df.loc[model_a_results_df["modelo"] == "DummyClassifier", "f1_score"].iloc[0])
model_a_best_f1 = float(model_a_results_df.iloc[0]["f1_score"])
model_a_supera_baseline = model_a_best_f1 > model_a_baseline_f1
print(f"Modelo A supera baseline? {model_a_supera_baseline} | F1 melhor: {model_a_best_f1:.4f} | F1 baseline: {model_a_baseline_f1:.4f}")
assert "DummyClassifier" in set(model_a_results_df["modelo"])
assert model_a_supera_baseline


## Modelo A: incerteza do F1 sem repetir treinamentos

A comparação entre os modelos já foi feita na validação do treino. Para não treinar novamente os mesmos modelos, a incerteza abaixo usa as previsões do modelo que foi bloqueado antes de consultar o teste.

O intervalo é calculado por bootstrap dos municípios: os municípios do teste são reamostrados, mantendo juntos os participantes do mesmo município. Essa medida é descritiva e não transforma o estudo em inferência causal, mas mostra a sensibilidade do F1 sem acrescentar novos ajustes de modelo.

In [ ]:
def cluster_bootstrap_f1(
    y_true: pd.Series,
    y_pred: np.ndarray,
    groups: pd.Series,
    n_bootstrap: int = F1_BOOTSTRAP_REPETITIONS,
) -> dict:
    """Estima um intervalo percentil do F1 reamostrando municípios inteiros."""
    frame = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred),
        "grupo": groups.reset_index(drop=True).fillna("NA").astype(str),
    })
    frame["tp"] = ((frame["y_true"] == 1) & (frame["y_pred"] == 1)).astype(int)
    frame["fp"] = ((frame["y_true"] == 0) & (frame["y_pred"] == 1)).astype(int)
    frame["fn"] = ((frame["y_true"] == 1) & (frame["y_pred"] == 0)).astype(int)
    counts = frame.groupby("grupo")[["tp", "fp", "fn"]].sum().to_numpy()
    rng = np.random.default_rng(RANDOM_STATE)
    sampled_groups = rng.integers(0, len(counts), size=(n_bootstrap, len(counts)))
    sampled_counts = counts[sampled_groups].sum(axis=1)
    denominator = 2 * sampled_counts[:, 0] + sampled_counts[:, 1] + sampled_counts[:, 2]
    f1_values = np.divide(
        2 * sampled_counts[:, 0],
        denominator,
        out=np.zeros_like(denominator, dtype=float),
        where=denominator > 0,
    )
    return {
        "f1_bootstrap_media": float(f1_values.mean()),
        "f1_bootstrap_desvio": float(f1_values.std()),
        "f1_ic_2_5": float(np.quantile(f1_values, 0.025)),
        "f1_ic_97_5": float(np.quantile(f1_values, 0.975)),
        "repeticoes": int(n_bootstrap),
        "grupos": int(len(counts)),
    }

In [ ]:
model_a_best_estimator = model_a_trained[model_a_best_name]
X_a_test_for_uncertainty = clean_feature_matrix(X_a_test, model_a_categorical_features, model_a_numeric_features)
model_a_test_score = get_score(model_a_best_estimator, X_a_test_for_uncertainty)
model_a_test_pred = (model_a_test_score >= 0.5).astype(int)
model_a_f1_uncertainty = cluster_bootstrap_f1(
    y_a_test,
    model_a_test_pred,
    model_a_test_df["CO_MUNICIPIO_PROVA"],
)
model_a_f1_uncertainty_df = pd.DataFrame([{
    "modelo": model_a_best_name,
    "f1_teste": float(f1_score(y_a_test, model_a_test_pred)),
    **model_a_f1_uncertainty,
}])
display(model_a_f1_uncertainty_df)
model_a_f1_uncertainty_df.to_csv(ARTIFACTS / "model_a_f1_uncertainty.csv", index=False)
display(Markdown(
    f"**F1 do modelo bloqueado:** {model_a_f1_uncertainty_df.loc[0, 'f1_teste']:.4f}. "
    f"Intervalo percentil por município: **{model_a_f1_uncertainty['f1_ic_2_5']:.4f} a "
    f"{model_a_f1_uncertainty['f1_ic_97_5']:.4f}**."
))

## Modelo A: matriz de confusão, erros e overfitting

A matriz de confusão ajuda a interpretar falsos positivos e falsos negativos. A comparação entre F1 de treino e teste indica se há indício de overfitting.


In [ ]:
def save_confusion_matrix(estimator, X_test: pd.DataFrame, y_test: pd.Series, title: str, path: Path) -> tuple[np.ndarray, pd.DataFrame]:
    """Salva a matriz de confusão e devolve uma tabela de erros por tipo."""
    y_pred = estimator.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Não alto", "Alto"]).plot(ax=ax, values_format="d", colorbar=False, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Classe prevista")
    ax.set_ylabel("Classe real")
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    errors = pd.DataFrame({"y_true": y_test.values, "y_pred": y_pred})
    errors["tipo_erro"] = np.select(
        [
            (errors["y_true"] == 1) & (errors["y_pred"] == 0),
            (errors["y_true"] == 0) & (errors["y_pred"] == 1),
            (errors["y_true"] == errors["y_pred"]),
        ],
        ["falso_negativo", "falso_positivo", "acerto"],
        default="indefinido",
    )
    return cm, errors


In [ ]:
model_a_best_estimator = model_a_trained[model_a_best_name]
X_a_test_clean = clean_feature_matrix(X_a_test, model_a_categorical_features, model_a_numeric_features)
model_a_cm, model_a_errors = save_confusion_matrix(
    model_a_best_estimator,
    X_a_test_clean,
    y_a_test,
    f"Matriz de confusão - Modelo A ({model_a_best_name})",
    ARTIFACTS / "model_a_confusion_matrix.png",
)
model_a_error_summary = model_a_errors["tipo_erro"].value_counts().rename_axis("tipo_erro").reset_index(name="quantidade")
model_a_error_summary["percentual"] = (model_a_error_summary["quantidade"] / model_a_error_summary["quantidade"].sum() * 100).round(2)
display(model_a_error_summary)
model_a_error_summary.to_csv(ARTIFACTS / "model_a_error_summary.csv", index=False)

model_a_train_f1 = float(model_a_results_long_df[(model_a_results_long_df["modelo"] == model_a_best_name) & (model_a_results_long_df["split"] == "treino_modelagem")]["f1_score"].iloc[0])
model_a_gap = model_a_train_f1 - model_a_best_f1
print(f"Modelo A - F1 treino: {model_a_train_f1:.4f}; F1 teste: {model_a_best_f1:.4f}; diferença: {model_a_gap:.4f}")


## Modelo A: interpretabilidade e leitura de desigualdade

A pergunta socioeducacional só fica respondida se mostrarmos **quais** variáveis pesam e em que direção. Como o melhor modelo do Experimento A é a `LogisticRegression`, lemos seus coeficientes (log-odds) já com os nomes das features após a codificação. Coeficiente positivo eleva a chance de alto desempenho; negativo reduz.

A leitura é associativa, não causal, mas conecta o modelo à narrativa de desigualdade: renda (`Q006`), dependência administrativa da escola e perfil socioeconômico aparecem entre os fatores mais influentes.


In [ ]:
final_model_a = model_a_best_estimator.named_steps["model"]
feature_names_a = model_a_best_estimator.named_steps["preprocessor"].get_feature_names_out()
clean_names_a = [name.split("__", 1)[-1] for name in feature_names_a]

if hasattr(final_model_a, "coef_"):
    importance_values = final_model_a.coef_[0]
    importance_kind = "coeficiente (log-odds)"
else:
    importance_values = final_model_a.feature_importances_
    importance_kind = "importancia (Gini)"

model_a_importance = pd.DataFrame({"feature": clean_names_a, "valor": importance_values})
model_a_importance["abs_valor"] = model_a_importance["valor"].abs()
model_a_importance = model_a_importance.sort_values("abs_valor", ascending=False).reset_index(drop=True)
model_a_importance.to_csv(ARTIFACTS / "model_a_feature_importance.csv", index=False)

print(f"Importancia usada: {importance_kind}")
display(model_a_importance.drop(columns="abs_valor").head(15))

socio_prefixes = ("Q006", "TP_ESCOLA", "TP_ENSINO", "TP_DEPENDENCIA_ADM_ESC", "TP_COR_RACA")
socio_importance = (
    model_a_importance[model_a_importance["feature"].str.startswith(socio_prefixes)]
    .drop_duplicates("feature")
    .sort_values("valor")
)

fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(socio_importance))))
coef_colors = np.where(socio_importance["valor"] >= 0, CHART_COLORS["green"], CHART_COLORS["red"])
ax.barh(socio_importance["feature"], socio_importance["valor"], color=coef_colors)
ax.axvline(0, color=CHART_COLORS["dark_gray"], linewidth=0.8)
ax.set_title("Modelo A: coeficientes de variaveis socioeconomicas e escolares")
ax.set_xlabel("Coeficiente (log-odds): positivo eleva a chance de alto desempenho")
ax.set_ylabel("Feature")
fig.tight_layout()
fig.savefig(ARTIFACTS / "model_a_socioeconomic_coefficients.png", dpi=160)
plt.show()

assert (ARTIFACTS / "model_a_feature_importance.csv").exists()


## Modelo A: erro por subgrupo (equidade descritiva)

A discussão ética fica mais forte com evidência, não só com prosa. A tabela abaixo mede, no conjunto de teste, a **taxa de falso negativo** (proporção de alunos de alto desempenho que o modelo deixa de identificar) e a **taxa de falso positivo** por tipo de escola e por cor/raça.

Diferenças grandes entre grupos indicam que a utilidade do modelo não é distribuída de forma uniforme — alerta direto para qualquer uso em política pública.


In [ ]:
def subgroup_error_table(data: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Calcula proporcao real de alto desempenho e taxas de erro por subgrupo."""
    rows = []
    for group_value, group_df in data.groupby(group_col, dropna=False):
        positives = group_df[group_df["y_true"] == 1]
        negatives = group_df[group_df["y_true"] == 0]
        rows.append({
            "grupo": f"{group_col}={group_value}",
            "n": len(group_df),
            "prop_alto_real": round(group_df["y_true"].mean(), 4),
            "taxa_falso_negativo": round((positives["y_pred"] == 0).mean(), 4) if len(positives) else np.nan,
            "taxa_falso_positivo": round((negatives["y_pred"] == 1).mean(), 4) if len(negatives) else np.nan,
        })
    return pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)


In [ ]:
y_pred_a_subgroup = model_a_best_estimator.predict(
    clean_feature_matrix(X_a_test, model_a_categorical_features, model_a_numeric_features)
)
subgroup_frame = pd.DataFrame({
    "TP_ESCOLA": model_a_test_df["TP_ESCOLA"].values,
    "TP_COR_RACA": model_a_test_df["TP_COR_RACA"].values,
    "y_true": y_a_test.values,
    "y_pred": y_pred_a_subgroup,
})

subgroup_error_school = subgroup_error_table(subgroup_frame, "TP_ESCOLA")
subgroup_error_race = subgroup_error_table(subgroup_frame, "TP_COR_RACA")
subgroup_error_model_a = pd.concat([subgroup_error_school, subgroup_error_race], ignore_index=True)
subgroup_error_model_a.to_csv(ARTIFACTS / "subgroup_error_model_a.csv", index=False)

display(Markdown("**TP_ESCOLA** (1 = nao respondeu, 2 = publica, 3 = privada)"))
display(subgroup_error_school)
display(Markdown("**TP_COR_RACA** (0 = nao declarado, 1 = Branca, 2 = Preta, 3 = Parda, 4 = Amarela, 5 = Indigena)"))
display(subgroup_error_race)

print(
    "Leitura: o modelo deixa de identificar uma fracao muito maior de alunos de alto desempenho "
    "da escola publica e de estudantes pretos, pardos e indigenas do que dos grupos de referencia."
)
assert (ARTIFACTS / "subgroup_error_model_a.csv").exists()


> **Limitações desta análise de equidade:**
>
> **1. Viés de seleção — 60,4% do test set não declarou escola.**
> `TP_ESCOLA = 1` ("não respondeu") representa **42.830 de 70.970 observações** no
> conjunto de teste. As taxas de FN e FP por escola pública/privada referem-se apenas
> aos **39,6% que declararam**. Perfis que não declaram escola (egressos, EJA, ensino
> em casa) podem ter distribuição de desempenho diferente — as taxas reportadas não
> generalizam para o total do teste.
>
> **2. Escola privada: FP = 66,9% — alerta adicional.**
> Além da baixa taxa de falso negativo (≈10,6%), o modelo classifica **~67% dos
> estudantes de escola privada *sem* alto desempenho real como "alto desempenho"**
> (TP_ESCOLA=3: FP = 0,668). Um sistema de seleção baseado neste modelo privilegiaria
> a escola privada por recall alto *e* FP alto simultaneamente. Qualquer uso prático
> exige revisão do limiar de decisão e consideração do custo assimétrico dos erros
> por grupo socioeconômico.

# Experimento B — Robustez Temporal

O Modelo B testa H2 e H3. Ele treina em 2022-2023 e avalia em 2024. Como 2024 não permite junção individual entre perfil e resultado, o perfil de 2024 entra apenas como agregado municipal.


## Agregados municipais de perfil

Os agregados municipais são calculados por `NU_ANO + CO_MUNICIPIO_PROVA`. Eles representam contexto municipal, não características individuais. Esse desenho permite usar informação de perfil em 2024 sem tentar reconstruir um relacionamento individual inexistente.


In [ ]:
def make_municipal_profile_aggregates(profile_df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Cria proporções municipais de variáveis de perfil sem junção individual em 2024."""
    keys = ["NU_ANO", "CO_MUNICIPIO_PROVA"]
    base = profile_df[keys].dropna().drop_duplicates().copy()
    group_size = profile_df.dropna(subset=keys).groupby(keys).size().rename("agg_profile_n")
    aggregates = group_size.reset_index()

    for col in columns:
        if col not in profile_df.columns:
            continue
        temp = profile_df[keys + [col]].dropna(subset=keys).copy()
        temp[col] = temp[col].astype("object").where(temp[col].notna(), "NA").astype(str)
        dummies = pd.get_dummies(temp[col], prefix=f"agg_{col}")
        temp_dummies = pd.concat([temp[keys], dummies], axis=1)
        sums = temp_dummies.groupby(keys).sum()
        props = sums.div(group_size, axis=0).reset_index()
        aggregates = aggregates.merge(props, on=keys, how="left")

    aggregate_cols = [col for col in aggregates.columns if col.startswith("agg_") and col != "agg_profile_n"]
    aggregates[aggregate_cols] = aggregates[aggregate_cols].fillna(0.0)
    return aggregates


In [ ]:
municipal_aggregates = make_municipal_profile_aggregates(df_profiles, AGGREGATE_PROFILE_COLUMNS)
municipal_aggregates.to_csv(ARTIFACTS / "municipal_profile_aggregates_2022_2024.csv", index=False)
display(municipal_aggregates.head())
print("Agregados municipais:", municipal_aggregates.shape)

coverage_2024 = (
    df_results[df_results["NU_ANO"] == EXTERNAL_TEST_YEAR][["NU_ANO", "CO_MUNICIPIO_PROVA"]]
    .drop_duplicates()
    .merge(municipal_aggregates[["NU_ANO", "CO_MUNICIPIO_PROVA", "agg_profile_n"]], on=["NU_ANO", "CO_MUNICIPIO_PROVA"], how="left")
)
coverage_2024["tem_agregado"] = coverage_2024["agg_profile_n"].notna()
coverage_2024.to_csv(ARTIFACTS / "municipal_aggregate_coverage_2024.csv", index=False)
display(coverage_2024)

assert coverage_2024["tem_agregado"].all(), "Algum municipio de resultado 2024 ficou sem agregado municipal."
assert coverage_2024["CO_MUNICIPIO_PROVA"].nunique() == 52


## Limitações dos Agregados Municipais

Agregados municipais introduzem risco de **falácia ecológica**, que ocorre quando conclusões sobre indivíduos são inferidas indevidamente a partir de estatísticas agregadas de grupos.

Neste notebook, uma variável como a proporção municipal de determinada faixa de `Q006` não significa que um participante específico tenha aquela renda. Também não permite afirmar que um estudante pertence ao perfil médio do município. As interpretações válidas são contextuais: municípios com certos perfis agregados podem estar associados a diferenças de desempenho, mas isso não descreve indivíduos nem estabelece causalidade.


## EDA temporal: o perfil municipal mudou em 2024?

A distribuição das **notas** de 2024 já aparece na EDA inicial (histograma por ano). O que ainda não foi verificado é a **comparabilidade do perfil agregado** entre 2022-2023 e 2024 — pergunta central para H2 por dois motivos distintos:

1. **Mudança de população:** se o perfil real dos participantes de 2024 for muito diferente, a queda do Modelo B pode refletir público novo, não instabilidade dos padrões.
2. **Mudança de schema:** o questionário de 2024 foi **renumerado** (vai só até Q023). Se uma pergunta mudou de significado mantendo o mesmo código, as features agregadas treinadas em 2022-2023 recebem, em 2024, proporções de **outra pergunta** — incompatibilidade semântica silenciosa de feature no teste externo.

A tabela compara, para cada proporção municipal (`agg_*`), a média simples entre municípios em 2024 contra 2022-2023, ranqueada pelo desvio absoluto. A leitura automática abaixo separa os dois fenômenos: categorias que **zeram** em 2024 são assinatura de mudança de codificação (uma população não abandona 6 faixas de renda de um ano para o outro); o drift das demais features mede a mudança real de perfil.

In [ ]:
def municipal_profile_drift(aggregates: pd.DataFrame) -> pd.DataFrame:
    """Compara o perfil municipal agregado de 2024 com a média 2022-2023, feature a feature."""
    feature_cols = [col for col in aggregates.columns if col.startswith("agg_") and col != "agg_profile_n"]
    base_hist = aggregates[aggregates["NU_ANO"].isin(TRAIN_YEARS)]
    base_2024 = aggregates[aggregates["NU_ANO"] == EXTERNAL_TEST_YEAR]
    rows = []
    for col in feature_cols:
        if not base_2024[col].notna().any():
            continue
        media_hist = float(base_hist[col].mean())
        media_2024 = float(base_2024[col].mean())
        rows.append({
            "feature": col,
            "media_2022_2023": round(media_hist, 4),
            "media_2024": round(media_2024, 4),
            "drift_abs": round(abs(media_2024 - media_hist), 4),
        })
    return pd.DataFrame(rows).sort_values("drift_abs", ascending=False).reset_index(drop=True)

In [ ]:
profile_drift_df = municipal_profile_drift(municipal_aggregates)
display(profile_drift_df.head(15))
profile_drift_df.to_csv(ARTIFACTS / "municipal_profile_drift_2024.csv", index=False)

q006_levels_by_year = df_profiles.groupby("NU_ANO")["Q006"].nunique().rename("categorias_distintas_Q006")
display(q006_levels_by_year.to_frame())

schema_suspects = profile_drift_df[
    (profile_drift_df["media_2024"] == 0.0) & (profile_drift_df["media_2022_2023"] > 0.01)
]
suspect_prefixes = sorted({feat.rsplit("_", 1)[0] for feat in schema_suspects["feature"]})
if suspect_prefixes:
    is_suspect = profile_drift_df["feature"].apply(lambda f: f.startswith(tuple(suspect_prefixes)))
else:
    is_suspect = pd.Series(False, index=profile_drift_df.index)

drift_populacional = profile_drift_df[~is_suspect]
drift_max_pop = float(drift_populacional["drift_abs"].max())
feature_max_pop = str(drift_populacional.iloc[0]["feature"]) if len(drift_populacional) else "-"

if len(schema_suspects) > 0:
    leitura_schema = (
        f"**{len(schema_suspects)} categorias agregadas zeram em 2024** (prefixos: {', '.join(suspect_prefixes)}) — "
        f"assinatura de **mudança de codificação**, não de população. A contagem de categorias distintas de `Q006` "
        f"por ano confirma: { {int(year): int(n) for year, n in q006_levels_by_year.items()} }. Em 2024, `Q006` é **outra pergunta** (binária), e as "
        f"features `agg_Q006_*` são **semanticamente incompatíveis no teste externo**. Por isso, esse bloco "
        f"é removido antes da seleção e do treinamento do Modelo B, sem consultar o desempenho em 2024."
    )
else:
    leitura_schema = "Nenhuma categoria agregada zera em 2024 — sem indício de mudança de codificação."

if drift_max_pop < 0.05:
    leitura_pop = (
        f"Excluído o bloco suspeito de schema, o drift máximo das demais features é **{drift_max_pop:.4f}** "
        f"(`{feature_max_pop}`) — o perfil **populacional** de 2024 é essencialmente estável. "
        f"A queda do Modelo B não é explicada por mudança de público."
    )
else:
    leitura_pop = (
        f"Mesmo fora do bloco suspeito, há drift de até **{drift_max_pop:.4f}** (`{feature_max_pop}`) — "
        f"parte da queda do Modelo B pode refletir mudança real de perfil."
    )

display(Markdown(f"{leitura_schema}\n\n{leitura_pop}"))

## Modelo B: preparação, target e features

O limiar P75 do Modelo B é calculado apenas com o treino temporal 2022-2023. O teste externo de 2024 não participa da definição do target nem da otimização de hiperparâmetros. As colunas `agg_Q006_*` são excluídas antes da modelagem porque `Q006` mudou de significado em 2024; esta é uma decisão de compatibilidade de schema, não uma escolha baseada no resultado do teste.


In [ ]:
model_b_base = df_results.merge(municipal_aggregates, on=["NU_ANO", "CO_MUNICIPIO_PROVA"], how="left")
model_b_train_df = model_b_base[model_b_base["NU_ANO"].isin(TRAIN_YEARS)].copy()
model_b_test_df = model_b_base[model_b_base["NU_ANO"] == EXTERNAL_TEST_YEAR].copy()
model_b_threshold = model_b_train_df[TARGET_SCORE_COLUMN].quantile(TARGET_QUANTILE)
for frame in [model_b_train_df, model_b_test_df]:
    frame[TARGET_COLUMN] = (frame[TARGET_SCORE_COLUMN] >= model_b_threshold).astype(int)

model_b_direct_features = [col for col in SCHOOL_COLUMNS + ["CO_MUNICIPIO_PROVA", "NO_MUNICIPIO_PROVA"] if col in model_b_base.columns]
model_b_incompatible_q006_features = sorted(col for col in municipal_aggregates.columns if col.startswith("agg_Q006"))
model_b_aggregate_features = [
    col for col in municipal_aggregates.columns
    if col.startswith("agg_") and col not in model_b_incompatible_q006_features
]
model_b_features = model_b_direct_features + model_b_aggregate_features
model_b_forbidden = set(PRESENCE_COLUMNS + SCORE_COLUMNS + REDACAO_COMP_COLUMNS + ["NU_ANO", TARGET_COLUMN, "SG_UF_PROVA", "CO_UF_PROVA"])
model_b_features = [col for col in model_b_features if col not in model_b_forbidden and not col.startswith(LEAKAGE_PREFIXES)]

model_b_numeric_features = [col for col in model_b_features if col.startswith("agg_")]
model_b_categorical_features = [col for col in model_b_features if col not in model_b_numeric_features]

X_b_train_full = model_b_train_df[model_b_features].copy()
y_b_train_full = model_b_train_df[TARGET_COLUMN].copy()
X_b_test = model_b_test_df[model_b_features].copy()
y_b_test = model_b_test_df[TARGET_COLUMN].copy()

model_b_split_summary = pd.DataFrame({
    "particao": ["treino_b_2022_2023", "teste_b_2024"],
    "linhas": [len(X_b_train_full), len(X_b_test)],
    "proporcao_target_1": [y_b_train_full.mean(), y_b_test.mean()],
})
display(Markdown(f"Limiar P75 do Modelo B calculado apenas no treino temporal 2022-2023: **{model_b_threshold:.2f}**."))
display(model_b_split_summary)

assert len(X_b_train_full) > 0 and len(X_b_test) > 0
assert TARGET_SCORE_COLUMN not in model_b_features
assert not any(col.startswith(LEAKAGE_PREFIXES) for col in model_b_features)
assert not any(col.startswith("agg_Q006") for col in model_b_features)
assert not X_b_test[model_b_numeric_features].isna().all(axis=None)


## Rastreabilidade do fluxo de dados

A tabela abaixo registra as dimensões em etapas críticas. Ela permite reconstruir como os microdados brutos chegam às bases usadas nos dois experimentos.


In [ ]:
result_metadata_df = pd.DataFrame(result_metadata)
profile_metadata_df = pd.DataFrame(profile_metadata)

data_flow_rows = []
for row in result_metadata_df.to_dict("records"):
    data_flow_rows.append({
        "etapa": "resultados_brutos_lidos",
        "ano": row["year"],
        "linhas": row["rows_seen"],
        "detalhe": row["csv_member"],
    })
    data_flow_rows.append({
        "etapa": "resultados_rj_presenca_mt_nota_valida",
        "ano": row["year"],
        "linhas": row["rows_kept_score_valid_rj"],
        "detalhe": "SG_UF_PROVA == RJ; TP_PRESENCA_MT == 1; NU_NOTA_MT válida",
    })
for row in profile_metadata_df.to_dict("records"):
    data_flow_rows.append({
        "etapa": "perfis_rj_lidos",
        "ano": row["year"],
        "linhas": row["rows_kept_profile_rj"],
        "detalhe": "SG_UF_PROVA == RJ na base de perfil/participantes",
    })

data_flow_rows.extend([
    {"etapa": "modelo_a_treino_2022_2023", "ano": "2022-2023", "linhas": len(X_a_train_full), "detalhe": "Split estratificado; P75 calculado apenas aqui."},
    {"etapa": "modelo_a_teste_2022_2023", "ano": "2022-2023", "linhas": len(X_a_test), "detalhe": "Teste interno não usado para definir P75."},
    {"etapa": "agregados_municipais", "ano": "2022-2024", "linhas": len(municipal_aggregates), "detalhe": "Proporções por NU_ANO + CO_MUNICIPIO_PROVA."},
    {"etapa": "cobertura_agregados_2024", "ano": 2024, "linhas": int(coverage_2024["tem_agregado"].sum()), "detalhe": "Municípios de resultado 2024 com agregado municipal."},
    {"etapa": "modelo_b_treino_2022_2023", "ano": "2022-2023", "linhas": len(X_b_train_full), "detalhe": "Treino temporal; P75 calculado apenas aqui."},
    {"etapa": "modelo_b_teste_externo_2024", "ano": 2024, "linhas": len(X_b_test), "detalhe": "Teste externo sem uso em target ou tuning."},
])

data_flow_trace = pd.DataFrame(data_flow_rows)
display(data_flow_trace)
data_flow_trace.to_csv(ARTIFACTS / "data_flow_trace.csv", index=False)
assert {"modelo_a_treino_2022_2023", "modelo_b_teste_externo_2024"}.issubset(set(data_flow_trace["etapa"]))


## Modelo B: auditoria das features

A auditoria documenta a separação entre features diretas e agregados municipais. O uso dos agregados é contextual e nunca individual.


In [ ]:
feature_audit_model_b = build_feature_audit(
    list(model_b_base.columns),
    model_b_features,
    model_b_forbidden,
    unavailable_columns=set(model_b_incompatible_q006_features),
)
feature_audit_model_b["tipo_feature"] = np.select(
    [feature_audit_model_b["coluna"].isin(model_b_numeric_features), feature_audit_model_b["coluna"].isin(model_b_categorical_features)],
    ["agregado_municipal", "direta_comparavel"],
    default="nao_usada",
)
display(feature_audit_model_b)
feature_audit_model_b.to_csv(ARTIFACTS / "feature_audit_model_b.csv", index=False)
assert set(model_b_features) == set(feature_audit_model_b.loc[feature_audit_model_b["status"] == "usada", "coluna"])


## Auditoria das variáveis excluídas

Esta auditoria consolida o que ficou fora dos pipelines. O objetivo é tornar explícitas as exclusões por vazamento, incompatibilidade de schema, identificadores, colunas constantes ou ausência de relação direta com o desenho do experimento.


In [ ]:
excluded_variables_audit = pd.concat([
    feature_audit_model_a.assign(experimento="Modelo A"),
    feature_audit_model_b.assign(experimento="Modelo B"),
], ignore_index=True)
excluded_variables_audit = excluded_variables_audit[excluded_variables_audit["status"] == "removida"].copy()
excluded_variables_audit.to_csv(ARTIFACTS / "excluded_variables_audit.csv", index=False)
display(excluded_variables_audit.head(40))
assert not excluded_variables_audit.empty
leakage_removed_a = set(feature_audit_model_a.loc[feature_audit_model_a["motivo"].str.contains("Vazamento", na=False), "coluna"])
leakage_removed_b = set(feature_audit_model_b.loc[feature_audit_model_b["motivo"].str.contains("Vazamento", na=False), "coluna"])
assert leakage_removed_a.isdisjoint(model_a_features)
assert leakage_removed_b.isdisjoint(model_b_features)


## Modelo B: treinamento, baseline temporal e avaliação externa

O baseline temporal é obrigatório: `DummyClassifier(strategy="most_frequent")`, treinado em 2022-2023 e avaliado no conjunto externo de 2024. A escolha do modelo é feita antes, usando 2022 para ajuste e 2023 para validação. A busca compara as configurações da RF com 80 árvores e reserva 120 árvores para um eventual ajuste final. Assim, 2024 permanece como avaliação externa e não participa da seleção.


In [ ]:
model_b_suite_start = time.perf_counter()
model_b_results_df, model_b_results_long_df, model_b_trained, model_b_best_name = run_model_suite(
    "Modelo B",
    X_b_train_full,
    y_b_train_full,
    X_b_test,
    y_b_test,
    model_b_categorical_features,
    model_b_numeric_features,
    "teste_externo_2024",
    model_b_train_df["NU_ANO"],
    selection_mode="temporal",
)
model_b_suite_seconds = time.perf_counter() - model_b_suite_start

display(Markdown("**Seleção temporal feita apenas no treino (2022 → 2023):**"))
display(model_b_trained["selection_results"])
model_b_trained["selection_results"].to_csv(ARTIFACTS / "model_b_training_selection.csv", index=False)
display(model_b_results_df.drop(columns=["confusion_matrix"]))
model_b_baseline_f1 = float(model_b_results_df.loc[model_b_results_df["modelo"] == "DummyClassifier", "f1_score"].iloc[0])
model_b_best_f1 = float(model_b_results_df.iloc[0]["f1_score"])
model_b_supera_baseline = model_b_best_f1 > model_b_baseline_f1
print(f"O modelo temporal supera um baseline temporal? {model_b_supera_baseline}")
print(f"F1 melhor modelo temporal: {model_b_best_f1:.4f}; F1 baseline temporal: {model_b_baseline_f1:.4f}")
assert "DummyClassifier" in set(model_b_results_df["modelo"])
assert model_b_supera_baseline


## Modelo B: matriz de confusão, erros e overfitting temporal

A avaliação externa em 2024 mede robustez temporal. A diferença entre F1 de treino e F1 externo sinaliza o quanto os padrões aprendidos permanecem úteis em outro ano.


In [ ]:
model_b_best_estimator = model_b_trained[model_b_best_name]
X_b_test_clean = clean_feature_matrix(X_b_test, model_b_categorical_features, model_b_numeric_features)
model_b_cm, model_b_errors = save_confusion_matrix(
    model_b_best_estimator,
    X_b_test_clean,
    y_b_test,
    f"Matriz de confusão - Modelo B ({model_b_best_name}) em 2024",
    ARTIFACTS / "model_b_confusion_matrix_2024.png",
)
model_b_error_summary = model_b_errors["tipo_erro"].value_counts().rename_axis("tipo_erro").reset_index(name="quantidade")
model_b_error_summary["percentual"] = (model_b_error_summary["quantidade"] / model_b_error_summary["quantidade"].sum() * 100).round(2)
display(model_b_error_summary)
model_b_error_summary.to_csv(ARTIFACTS / "model_b_error_summary_2024.csv", index=False)

model_b_train_f1 = float(model_b_results_long_df[(model_b_results_long_df["modelo"] == model_b_best_name) & (model_b_results_long_df["split"] == "treino_modelagem")]["f1_score"].iloc[0])
model_b_gap = model_b_train_f1 - model_b_best_f1
print(f"Modelo B - F1 treino: {model_b_train_f1:.4f}; F1 externo 2024: {model_b_best_f1:.4f}; diferença: {model_b_gap:.4f}")


## Modelo B: leitura operacional honesta

Superar o `DummyClassifier(strategy="most_frequent")` é um teste fraco: esse baseline prevê sempre "não alto" e tem F1 = 0 por construção. Antes de comemorar, é preciso olhar o que o Modelo B faz na prática em 2024.


In [ ]:
tn_b, fp_b = int(model_b_cm[0][0]), int(model_b_cm[0][1])
fn_b, tp_b = int(model_b_cm[1][0]), int(model_b_cm[1][1])
total_b = tn_b + fp_b + fn_b + tp_b
pred_pos_b = fp_b + tp_b
precision_b = tp_b / (tp_b + fp_b) if (tp_b + fp_b) else float("nan")
recall_b = tp_b / (tp_b + fn_b) if (tp_b + fn_b) else float("nan")

print(f"Modelo B marca como 'alto desempenho': {pred_pos_b:,} de {total_b:,} ({pred_pos_b / total_b:.1%}).")
print(f"Precision = {precision_b:.3f} | Recall = {recall_b:.3f}")
print(f"Falsos positivos: {fp_b:,} ({fp_b / total_b:.1%} de todos os participantes de 2024).")
print(
    "Conclusao operacional: com AUC ~0.66 ha sinal fraco de ranqueamento, mas no limiar padrao o "
    "Modelo B rotula a maioria dos estudantes como 'alto' e tem baixa precisao. Serve como score "
    "contextual agregado, nao como classificador individual."
)


## Avaliação integrada sem comparação injusta

O Modelo A e o Modelo B são analisados em conjunto para responder à tese do estudo, mas não são ranqueados como se fossem o mesmo experimento. O Modelo A tem maior riqueza individual; o Modelo B tem maior rigor temporal.


In [ ]:
model_a_summary = model_a_results_df.assign(experimento="Modelo A - socioeducacional individual", avaliacao="teste interno 2022-2023")
model_b_summary = model_b_results_df.assign(experimento="Modelo B - robustez temporal", avaliacao="teste externo 2024")
combined_results = pd.concat([model_a_summary, model_b_summary], ignore_index=True)
metric_cols = ["experimento", "avaliacao", "modelo", "accuracy", "precision", "recall", "f1_score", "roc_auc", "training_seconds"]
display(combined_results[metric_cols])

combined_results[metric_cols].to_csv(ARTIFACTS / "model_results.csv", index=False)
model_a_results_df[metric_cols[2:]].to_csv(ARTIFACTS / "model_a_individual_results.csv", index=False)
model_b_results_df[metric_cols[2:]].to_csv(ARTIFACTS / "model_b_temporal_results.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
model_a_f1 = model_a_results_df.set_index("modelo")["f1_score"]
model_b_f1 = model_b_results_df.set_index("modelo")["f1_score"]
model_a_f1.plot(kind="bar", ax=axes[0], title="Modelo A: F1 no teste 2022-2023", color=[MODEL_COLORS.get(model, CHART_COLORS["gray"]) for model in model_a_f1.index])
model_b_f1.plot(kind="bar", ax=axes[1], title="Modelo B: F1 no teste externo 2024", color=[MODEL_COLORS.get(model, CHART_COLORS["gray"]) for model in model_b_f1.index])
for ax in axes:
    ax.set_ylabel("F1-score")
    ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(ARTIFACTS / "f1_by_experiment_not_direct_comparison.png", dpi=160)
plt.show()

print("Os dois modelos respondem perguntas diferentes e não devem ser interpretados como uma competição direta entre algoritmos.")


## Ganho real sobre um baseline honesto

O baseline `most_frequent` usado acima tem F1 = 0 por construção, o que torna a frase "supera o baseline" pouco informativa. Aqui usamos um baseline mais justo — `DummyClassifier(strategy="stratified")`, que respeita a prevalência da classe positiva — e reportamos o ganho absoluto e percentual de cada experimento.


In [ ]:
def fair_baseline_f1(y_train: pd.Series, y_test: pd.Series) -> float:
    """F1 de um baseline estratificado que respeita a prevalencia da classe positiva."""
    baseline = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
    baseline.fit(np.zeros((len(y_train), 1)), y_train)
    return float(f1_score(y_test, baseline.predict(np.zeros((len(y_test), 1)))))


In [ ]:
f1_fair_baseline_a = fair_baseline_f1(y_a_train_full, y_a_test)
f1_fair_baseline_b = fair_baseline_f1(y_b_train_full, y_b_test)

honest_gain = pd.DataFrame([
    {
        "experimento": "Modelo A (teste interno 2022-2023)",
        "melhor_modelo": model_a_best_name,
        "f1_melhor": round(model_a_best_f1, 4),
        "f1_baseline_most_frequent": round(model_a_baseline_f1, 4),
        "f1_baseline_estratificado": round(f1_fair_baseline_a, 4),
        "ganho_abs_vs_estratificado": round(model_a_best_f1 - f1_fair_baseline_a, 4),
        "ganho_rel_vs_estratificado_pct": round((model_a_best_f1 - f1_fair_baseline_a) / f1_fair_baseline_a * 100, 1),
    },
    {
        "experimento": "Modelo B (teste externo 2024)",
        "melhor_modelo": model_b_best_name,
        "f1_melhor": round(model_b_best_f1, 4),
        "f1_baseline_most_frequent": round(model_b_baseline_f1, 4),
        "f1_baseline_estratificado": round(f1_fair_baseline_b, 4),
        "ganho_abs_vs_estratificado": round(model_b_best_f1 - f1_fair_baseline_b, 4),
        "ganho_rel_vs_estratificado_pct": round((model_b_best_f1 - f1_fair_baseline_b) / f1_fair_baseline_b * 100, 1),
    },
])
display(honest_gain)
honest_gain.to_csv(ARTIFACTS / "honest_baseline_gain.csv", index=False)
print(
    "Leitura: contra um baseline honesto, o Modelo A entrega ganho grande e o Modelo B um ganho "
    "moderado. Nenhum dos dois e trivial, mas o Modelo B e bem mais modesto do que a comparacao "
    "com o baseline degenerado sugeria."
)
assert (ARTIFACTS / "honest_baseline_gain.csv").exists()


## A otimização de hiperparâmetros trouxe benefício real?

O `GridSearchCV` ajustou a Random Forest usando apenas a validação do treino. A comparação abaixo responde se a configuração otimizada superou a RF simples nessa validação, sem usar o teste para decidir.


In [ ]:
def optimization_benefit(experiment: str, results_df: pd.DataFrame, selection_df: pd.DataFrame) -> dict:
    """Compara modelos na validação do treino e mantém o teste apenas como resultado descritivo."""
    selection = selection_df.set_index("modelo")
    rf_rows = selection_df[selection_df["modelo"].str.startswith("RandomForest")]
    rf_simple_cv = float(selection.loc["RandomForestClassifier", "f1_validacao_media"])
    rf_best_row = rf_rows.sort_values("f1_validacao_media", ascending=False).iloc[0]
    lr_cv = float(selection.loc["LogisticRegression", "f1_validacao_media"])
    selected_name = str(selection_df.loc[selection_df["selecionado"], "modelo"].iloc[0])
    f1_test_by_model = results_df.set_index("modelo")["f1_score"]
    return {
        "experimento": experiment,
        "f1_validacao_logreg": round(lr_cv, 4),
        "f1_validacao_rf_simples": round(rf_simple_cv, 4),
        "f1_validacao_melhor_rf": round(float(rf_best_row["f1_validacao_media"]), 4),
        "melhor_configuracao_rf": str(rf_best_row["parametros"]),
        "otim_superou_rf_simples_na_validacao": bool(float(rf_best_row["f1_validacao_media"]) > rf_simple_cv),
        "modelo_selecionado_no_treino": selected_name,
        "f1_teste_modelo_selecionado": round(float(f1_test_by_model[selected_name]), 4),
    }


In [ ]:
optimization_summary = pd.DataFrame([
    optimization_benefit("Modelo A", model_a_results_df, model_a_trained["selection_results"]),
    optimization_benefit("Modelo B", model_b_results_df, model_b_trained["selection_results"]),
])
display(optimization_summary)
optimization_summary.to_csv(ARTIFACTS / "optimization_benefit_summary.csv", index=False)
print("A decisão sobre o modelo foi tomada pelos resultados de validação acima; o teste não alterou essa escolha.")
assert (ARTIFACTS / "optimization_benefit_summary.csv").exists()


## Por que H2 não é plenamente testável neste desenho

O Modelo A aprende com atributos **individuais**; o Modelo B, com **agregados municipais**. Logo, a queda de F1 (de ~0,58 para ~0,40) mistura dois efeitos que este desenho **não separa**:

1. **Perda de granularidade** — trocar features individuais por contexto municipal, por si só, derruba o desempenho mesmo sem nenhuma mudança temporal;
2. **Drift temporal real** — eventuais mudanças na relação entre variáveis e desempenho entre 2022-2023 e 2024.

A impossibilidade de junção individual em 2024 é uma limitação real dos microdados, não uma escolha. Por isso H2 é respondida apenas de forma parcial e como **proxy fraco**: há sinal contextual remanescente em 2024, mas não se pode atribuir a queda à estabilidade (ou instabilidade) dos padrões individuais. Próximo passo honesto: treinar um modelo restrito às features municipais também no teste interno de 2022-2023, isolando quanto da queda vem da granularidade e quanto vem do ano.


## Decompondo a queda: granularidade ou tempo?

A seção anterior mostrou que a queda de F1 entre o Modelo A (0,58 interno) e o Modelo B (0,40 externo) mistura dois efeitos que o desenho original não separa: a **perda de granularidade** (trocar atributos individuais por agregados municipais) e o **drift temporal** (2024 ser um ano diferente).

O experimento abaixo separa os dois fatores: treina uma `LogisticRegression` com as **mesmas features do Modelo B** (diretas comparáveis + agregados municipais), mas em um **split interno 80/20 de 2022-2023** — ou seja, sem mudar de ano. A disciplina anti-vazamento é idêntica à do Modelo A: P75 recalculado apenas no treino interno, teste intocado.

A leitura é direta:

- **F1 individual interno → F1 municipal interno** = custo da granularidade (mesmo período);
- **F1 municipal interno → F1 municipal externo 2024** = drift temporal puro (mesmas features).

*Ressalva:* a comparação não é perfeitamente simétrica — as features diretas do Modelo B (atributos de escola) diferem das do Modelo A (perfil + questionário individual) — mas isola o fator dominante da queda, que é o que a hipótese H2 precisa para ser interpretável.

In [ ]:
def h2_decomposition_experiment(data: pd.DataFrame, categorical_features: list[str], numeric_features: list[str]) -> dict:
    """Avalia as features municipais em split interno 2022-2023 para isolar o efeito da granularidade."""
    internal_train, internal_test = train_test_split(
        data,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=data["NU_ANO"].astype(str),
    )
    threshold = internal_train[TARGET_SCORE_COLUMN].quantile(TARGET_QUANTILE)
    y_train = (internal_train[TARGET_SCORE_COLUMN] >= threshold).astype(int)
    y_test = (internal_test[TARGET_SCORE_COLUMN] >= threshold).astype(int)
    features = categorical_features + numeric_features
    X_train = clean_feature_matrix(internal_train[features], categorical_features, numeric_features)
    X_test = clean_feature_matrix(internal_test[features], categorical_features, numeric_features)
    X_train, y_train = sample_training_data(X_train, y_train)
    pipeline = make_pipeline(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
        categorical_features,
        numeric_features,
    )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    return {
        "f1": float(f1_score(y_test, y_pred)),
        "roc_auc": float(roc_auc_score(y_test, y_proba)),
        "threshold_treino_interno": float(threshold),
        "linhas_teste_interno": int(len(X_test)),
    }

In [ ]:
h2_internal_municipal = h2_decomposition_experiment(model_b_train_df, model_b_categorical_features, model_b_numeric_features)

f1_individual_interno = float(model_a_results_df.loc[model_a_results_df["modelo"] == model_a_best_name, "f1_score"].iloc[0])
f1_municipal_interno = h2_internal_municipal["f1"]
f1_municipal_externo = float(model_b_results_df.loc[model_b_results_df["modelo"] == model_b_best_name, "f1_score"].iloc[0])

h2_decomposition_df = pd.DataFrame([
    {"cenario": "Individual, teste interno 2022-2023 (Modelo A)", "features": "individuais", "periodo_teste": "2022-2023", "f1": round(f1_individual_interno, 4)},
    {"cenario": "Municipal, teste interno 2022-2023 (novo)", "features": "municipais", "periodo_teste": "2022-2023", "f1": round(f1_municipal_interno, 4)},
    {"cenario": "Municipal, teste externo 2024 (Modelo B)", "features": "municipais", "periodo_teste": "2024", "f1": round(f1_municipal_externo, 4)},
])
display(h2_decomposition_df)
h2_decomposition_df.to_csv(ARTIFACTS / "h2_decomposition_internal_municipal.csv", index=False)

queda_granularidade = f1_individual_interno - f1_municipal_interno
queda_temporal = f1_municipal_interno - f1_municipal_externo
queda_total = f1_individual_interno - f1_municipal_externo
share_granularidade = (queda_granularidade / queda_total * 100) if queda_total > 0 else float("nan")

display(Markdown(
    f"**Decomposição da queda total de F1 ({queda_total:.4f}):** "
    f"a troca de features individuais por agregados municipais custa **{queda_granularidade:.4f}** "
    f"({share_granularidade:.0f}% da queda), ainda dentro de 2022-2023; "
    f"a passagem para 2024 com as mesmas features municipais custa **{queda_temporal:.4f}** "
    f"({100 - share_granularidade:.0f}% da queda). "
    f"AUC do cenário municipal interno: **{h2_internal_municipal['roc_auc']:.4f}**. "
    f"Conclusão: H2 deixa de ser um proxy não-interpretável — a parcela atribuível a drift temporal "
    f"está agora separada da perda de granularidade."
))

### Decisão de schema: exclusão das features Q006 incompatíveis

A EDA temporal mostrou que `Q006` muda de significado em 2024 (17 faixas de renda em 2022-2023 e uma pergunta binária em 2024). Por isso, o bloco `agg_Q006_*` foi removido **antes** da seleção e do treinamento do Modelo B. A exclusão é determinada pelo dicionário e pela incompatibilidade do schema, sem comparar alternativas no teste externo.

In [ ]:
model_b_schema_exclusion_df = pd.DataFrame([{
    "bloco_removido": "agg_Q006_*",
    "quantidade_colunas": len(model_b_incompatible_q006_features),
    "momento_da_decisao": "antes da seleção e do treinamento",
    "motivo": "Q006 possui significado incompatível em 2024",
}])
display(model_b_schema_exclusion_df)
model_b_schema_exclusion_df.to_csv(ARTIFACTS / "model_b_schema_exclusion_q006.csv", index=False)
assert not any(col.startswith("agg_Q006") for col in model_b_features)

## Evidências das Hipóteses

As hipóteses são avaliadas dentro do desenho ao qual pertencem. A tabela resume evidência, métrica, artefato de suporte e conclusão no escopo do MVP.


In [ ]:
model_a_best_row = model_a_results_df.iloc[0].to_dict()
model_b_best_row = model_b_results_df.iloc[0].to_dict()

hypothesis_evidence_summary = pd.DataFrame([
    {
        "hipotese": "H1",
        "pergunta": "Variáveis socioeducacionais e escolares ajudam a prever alto desempenho?",
        "evidencia": f"Modelo A: {model_a_best_name} com F1={model_a_best_f1:.3f} e ROC-AUC={model_a_best_row['roc_auc']:.3f}; baseline F1={model_a_baseline_f1:.3f}.",
        "artefatos": "model_a_individual_results.csv; model_a_confusion_matrix.png; feature_audit_model_a.csv",
        "conclusao": "Suportada no teste interno 2022-2023, sem interpretação causal.",
    },
    {
        "hipotese": "H2",
        "pergunta": "Parte dos padrões aprendidos em 2022-2023 permanece útil em 2024?",
        "evidencia": f"Modelo B: {model_b_best_name} com F1 externo={model_b_best_f1:.3f} e ROC-AUC={model_b_best_row['roc_auc']:.3f}; cobertura municipal 2024 completa em {coverage_2024['CO_MUNICIPIO_PROVA'].nunique()} municípios.",
        "artefatos": "model_b_temporal_results.csv; model_b_confusion_matrix_2024.png; municipal_aggregate_coverage_2024.csv",
        "conclusao": "Parcialmente suportada: há sinal em 2024, mas com perda de granularidade individual e risco ecológico.",
    },
    {
        "hipotese": "H3",
        "pergunta": "Modelos supervisionados superam baseline ingênuo nos dois desenhos?",
        "evidencia": f"Modelo A supera baseline: {model_a_supera_baseline}; Modelo B supera baseline temporal: {model_b_supera_baseline}.",
        "artefatos": "model_results.csv; f1_by_experiment_not_direct_comparison.png",
        "conclusao": "Suportada por F1 dentro de cada experimento, sem comparação direta entre A e B.",
    },
])

display(hypothesis_evidence_summary)
hypothesis_evidence_summary.to_csv(ARTIFACTS / "hypothesis_evidence_summary.csv", index=False)
assert set(hypothesis_evidence_summary["hipotese"]) == {"H1", "H2", "H3"}


## Como não interpretar estes resultados

Duas leituras devem ser evitadas.

Primeiro, o Modelo A e o Modelo B não são versões equivalentes do mesmo experimento. O Modelo A usa atributos individuais e responde à pergunta socioeducacional principal em 2022-2023. O Modelo B usa atributos comparáveis no tempo e agregados municipais para testar robustez temporal em 2024. Portanto, as métricas devem ser interpretadas dentro de cada desenho, não como uma disputa direta entre modelos.

Segundo, os resultados não demonstram causalidade. O notebook usa dados observacionais e modelos preditivos; por isso, identifica associações úteis para previsão e interpretação, mas não prova que uma variável socioeducacional cause alto ou baixo desempenho. Afirmações causais exigiriam outro desenho de pesquisa, controles adicionais e estratégia de identificação apropriada.

## Validade externa

**Outros estados:** os modelos não devem ser aplicados diretamente a outros estados sem nova validação. O recorte RJ tem composição, oferta escolar e distribuição municipal próprias.

**Anos futuros:** o Modelo B testa 2024, mas isso não garante validade em anos futuros. Mudanças de prova, política pública, perfil de participantes e schema dos microdados podem alterar a relação entre variáveis e desempenho.

**Outros contextos educacionais:** os modelos não devem ser transferidos diretamente para outros exames, escolas ou políticas de avaliação. O ENEM mede um contexto específico e usa escala própria.

A validade externa exige revalidação periódica, auditoria de vieses por grupo, monitoramento de mudança de distribuição e revisão pedagógica antes de qualquer uso prático.


## Limitações metodológicas e ética

O MVP usa dados observacionais, portanto não permite inferência causal. Mesmo quando variáveis socioeducacionais são preditivas, isso não significa que elas causem alto ou baixo desempenho.

O uso de agregados municipais no Modelo B é contextual e sujeito à falácia ecológica. Não se deve concluir que um participante individual tem as características médias do município.

O modelo não deve ser usado para rotular estudantes ou tomar decisões individuais. O valor do projeto está em demonstrar fluxo técnico de ML, avaliar padrões agregados e discutir desigualdades educacionais com cuidado.


## Viés de seleção: 60,4% do teste não declarou escola

A análise de equidade por tipo de escola (seção anterior) é tecnicamente correta, mas cobre **apenas 39,6% do conjunto de teste**. A distribuição real de `TP_ESCOLA` no conjunto de teste do Modelo A é:

| TP_ESCOLA | Rótulo | n (teste) | % do total de teste |
|---|---|---:|---:|
| 1 | Não respondeu | 42.830 | **60,4%** |
| 2 | Escola pública | 18.251 | 25,7% |
| 3 | Escola privada | 9.889 | 13,9% |

**Por que 60% não declararam escola?** Três hipóteses plausíveis:

1. **Treineiros e egressos** — quem já concluiu o ensino médio há mais de 1 ano pode não preencher esse campo, pois não tem vínculo escolar ativo.
2. **EJA e ensino informal** — participantes da Educação de Jovens e Adultos têm trajetória que pode não se enquadrar nos tipos disponíveis (pública/privada).
3. **Omissão voluntária** — o campo é facultativo para alguns perfis de inscrição.

**Implicação para a leitura das taxas de equidade:** as taxas de falso negativo (FN=50% pública vs. FN=10% privada) referem-se *apenas aos estudantes que declararam escola*. Se os 60% não-declarantes tiverem distribuição de desempenho diferente — plausível, dado que incluem egressos e perfis atípicos — as taxas de erro por grupo não generalizam para o universo total de participantes.

**O que isso não invalida:** a desigualdade *dentro do grupo que declarou escola* é real e rastreável. O modelo discrimina pior para escolas públicas — esse padrão independe dos não-declarantes. O caveat é sobre generalização, não sobre a existência do padrão de desigualdade.

## Exportação dos resultados

Esta etapa salva os principais resultados em `artifacts/`, incluindo métricas, parâmetros dos modelos, variáveis usadas, auditorias de vazamento e evidências citadas na conclusão. Esses arquivos permitem revisar os números sem depender apenas dos outputs visuais do notebook.


In [ ]:
def json_ready_records(df: pd.DataFrame) -> list[dict]:
    """Converte um DataFrame em registros JSON serializáveis."""
    return json.loads(df.to_json(orient="records"))


In [ ]:
performance_profile_df = pd.DataFrame([
    {
        "experimento": "Modelo A",
        "estrategia_selecao": f"StratifiedKFold({MODEL_SELECTION_FOLDS})",
        "linhas_treino_completas": len(X_a_train_full),
        "linhas_busca_rf": model_a_trained["X_train_modeling_rows"],
        "linhas_lr_final": model_a_trained["X_train_logistic_rows"],
        "candidatos_rf": 3,
        "folds_selecao": MODEL_SELECTION_FOLDS,
        "fits_principais_estimados": 14,
        "arvores_estimadas": 720 + RF_FINAL_N_ESTIMATORS * int(model_a_trained["selected_rf_refit_performed"]),
        "segundos_busca_rf": model_a_trained["selection_search_seconds"],
        "segundos_suite": model_a_suite_seconds,
    },
    {
        "experimento": "Modelo B",
        "estrategia_selecao": "2022_para_2023",
        "linhas_treino_completas": len(X_b_train_full),
        "linhas_busca_rf": model_b_trained["X_train_modeling_rows"],
        "linhas_lr_final": model_b_trained["X_train_logistic_rows"],
        "candidatos_rf": 3,
        "folds_selecao": 1,
        "fits_principais_estimados": 6,
        "arvores_estimadas": 240 + RF_FINAL_N_ESTIMATORS * int(model_b_trained["selected_rf_refit_performed"]),
        "segundos_busca_rf": model_b_trained["selection_search_seconds"],
        "segundos_suite": model_b_suite_seconds,
    },
])
performance_profile_df["jobs_rf"] = MODEL_PARALLEL_JOBS
performance_profile_df["jobs_busca"] = 1
performance_profile_df.to_csv(ARTIFACTS / "performance_profile.csv", index=False)
(ARTIFACTS / "performance_profile.json").write_text(
    performance_profile_df.to_json(orient="records", indent=2),
    encoding="utf-8",
)

model_a_payload = {
    "experiment": "Modelo A - Socioeducacional Individual",
    "question": "Variáveis socioeducacionais individuais em 2022-2023 ajudam a prever alto desempenho em Matemática?",
    "target_threshold_train_only": float(model_a_threshold),
    "features_used": model_a_features,
    "categorical_features": model_a_categorical_features,
    "numeric_features": model_a_numeric_features,
    "train_rows_full": int(len(X_a_train_full)),
    "train_rows_modeling": int(model_a_trained["X_train_modeling_rows"]),
    "test_rows": int(len(X_a_test)),
    "results": json_ready_records(model_a_results_df),
    "train_results": json_ready_records(model_a_results_long_df),
    "selection_results": json_ready_records(model_a_trained["selection_results"]),
    "best_model_within_experiment": model_a_best_name,
    "best_model_selected_without_test": True,
    "best_params": model_a_trained["final_model_params"],
    "rf_search_best_params": model_a_trained["grid_search_best_params"],
    "f1_uncertainty": model_a_f1_uncertainty,
    "supera_baseline": bool(model_a_supera_baseline),
}
model_b_payload = {
    "experiment": "Modelo B - Robustez Temporal",
    "question": "Padrões aprendidos em 2022-2023 se mantêm em 2024 usando features comparáveis e agregados municipais?",
    "target_threshold_train_only": float(model_b_threshold),
    "features_used": model_b_features,
    "categorical_features": model_b_categorical_features,
    "numeric_features": model_b_numeric_features,
    "train_rows_full": int(len(X_b_train_full)),
    "train_rows_modeling": int(model_b_trained["X_train_modeling_rows"]),
    "external_test_rows_2024": int(len(X_b_test)),
    "results": json_ready_records(model_b_results_df),
    "train_results": json_ready_records(model_b_results_long_df),
    "selection_results": json_ready_records(model_b_trained["selection_results"]),
    "best_model_within_experiment": model_b_best_name,
    "best_model_selected_without_external_test": True,
    "best_params": model_b_trained["final_model_params"],
    "rf_search_best_params": model_b_trained["grid_search_best_params"],
    "schema_features_removed": model_b_incompatible_q006_features,
    "supera_baseline_temporal": bool(model_b_supera_baseline),
    "municipal_aggregate_warning": "Agregados municipais sao contexto; nao representam individuos.",
}
combined_payload = {
    "source": "INEP Microdados ENEM",
    "source_url": "https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem",
    "execution_data_source_mode": DATA_SOURCE_MODE,
    "colab_cache_base_url": PUBLIC_CACHE_BASE_URL if IN_COLAB else None,
    "years": sorted(EXPECTED_YEARS),
    "recorte": "SG_UF_PROVA == RJ",
    "filters": ["TP_PRESENCA_MT == 1", "NU_NOTA_MT valida"],
    "hypotheses": {
        "H1": "Variáveis socioeducacionais e escolares possuem capacidade preditiva relevante.",
        "H2": "Parte dos padrões de 2022-2023 permanece estável em 2024.",
        "H3": "Modelos supervisionados superam baseline ingênuo nos dois desenhos.",
    },
    "methodological_note": "Modelo A e Modelo B nao sao competicao direta; respondem perguntas diferentes.",
    "model_a": model_a_payload,
    "model_b": model_b_payload,
    "artifacts": [
        "model_results.csv",
        "model_a_individual_results.csv",
        "model_a_individual_results.json",
        "model_b_temporal_results.csv",
        "model_b_temporal_results.json",
        "municipal_profile_aggregates_2022_2024.csv",
        "municipal_aggregate_coverage_2024.csv",
        "feature_audit_model_a.csv",
        "feature_audit_model_b.csv",
        "model_a_training_selection.csv",
        "model_b_training_selection.csv",
        "model_a_f1_uncertainty.csv",
        "model_b_schema_exclusion_q006.csv",
        "performance_profile.csv",
        "performance_profile.json",
    ],
}

(ARTIFACTS / "model_a_individual_results.json").write_text(json.dumps(model_a_payload, indent=2, ensure_ascii=False), encoding="utf-8")
(ARTIFACTS / "model_b_temporal_results.json").write_text(json.dumps(model_b_payload, indent=2, ensure_ascii=False), encoding="utf-8")
(ARTIFACTS / "model_results.json").write_text(json.dumps(combined_payload, indent=2, ensure_ascii=False), encoding="utf-8")
municipal_aggregates.to_csv(ARTIFACTS / "municipal_profile_aggregates_2022_2024.csv", index=False)
coverage_2024.to_csv(ARTIFACTS / "municipal_aggregate_coverage_2024.csv", index=False)

for name in combined_payload["artifacts"]:
    assert (ARTIFACTS / name).exists(), f"Artefato ausente: {name}"
print("Artefatos principais salvos em artifacts/.")


## Conclusão final

O MVP respondeu à pergunta central com duas evidências complementares. A base final de resultados elegíveis contém **557.163 participantes do ENEM/RJ** com presença em Matemática e `NU_NOTA_MT` válida, documentados em `artifacts/data_flow_trace.csv` e `artifacts/dataset_summary_by_year.csv`.

**Modelo A — pergunta socioeducacional individual.** O experimento A avaliou se atributos individuais, escolares e de questionário em 2022-2023 ajudam a prever alto desempenho. A **LogisticRegression** foi selecionada por validação cruzada apenas no treino e alcançou F1-score de **0,587** e ROC-AUC de **0,815** no teste interno. O baseline `most_frequent` teve F1-score de **0,000**; a comparação mais informativa é contra o baseline estratificado, com F1-score de **0,250**, sobre o qual o modelo entrega ganho relativo de **134,5%**. O bootstrap por município produziu intervalo percentil de F1 entre **0,518 e 0,610**. A hipótese H1 é suportada no escopo do MVP. As evidências estão em `model_a_individual_results.csv`, `model_a_training_selection.csv`, `model_a_f1_uncertainty.csv` e `model_a_confusion_matrix.png`.

**Modelo B — pergunta de robustez temporal.** O experimento B treinou em 2022-2023 e avaliou em 2024 usando variáveis comparáveis e agregados municipais. A **LogisticRegression** foi selecionada na validação temporal 2022 → 2023, antes da avaliação externa, e alcançou F1-score de **0,426** e ROC-AUC de **0,682** em 2024. O baseline temporal `most_frequent` teve F1-score de **0,000**; contra o baseline estratificado, com F1-score de **0,237**, o ganho relativo foi de **79,9%**. H2 e H3 recebem suporte parcial no teste temporal. A cobertura municipal de 2024 foi completa para **52 municípios**, conforme `municipal_aggregate_coverage_2024.csv`.

Os dois experimentos não devem ser lidos como competição direta entre algoritmos. O Modelo A tem maior valor interpretativo individual; o Modelo B testa generalização temporal sob a limitação real de schema de 2024. Como `Q006` muda de significado nesse ano, as 17 features `agg_Q006_*` foram excluídas antes da seleção e do treinamento. Assim, o resultado externo de **0,426** já corresponde ao desenho compatível de schema, sem escolher a alternativa pelo desempenho em 2024.

As principais limitações são: ausência de inferência causal, recorte restrito ao RJ, mudança de schema em 2024, impossibilidade de join individual entre participantes e resultados de 2024, uso de agregados municipais sujeito à falácia ecológica e necessidade de validação externa antes de aplicar a solução em outros estados, anos futuros ou contextos educacionais.

Como continuidade natural deste trabalho, faria sentido observar se os mesmos padrões aparecem em outros estados e em novas edições do ENEM. Também seria importante aprofundar a leitura dos erros por grupo e documentar com mais detalhe o dicionário socioeconômico, porque parte relevante da interpretação depende de entender bem o que cada variável representa.


## Checklist final

- [x] Notebook executa do início ao fim no fluxo validado.
- [x] Fonte oficial, anos, recorte geográfico e filtros estão documentados.
- [x] O carregamento público para Colab usa bases derivadas dos microdados oficiais, sem upload manual, login, token ou chave de API.
- [x] Dados brutos do ENEM não são versionados no repositório.
- [x] Target de alto desempenho em Matemática está definido e justificado.
- [x] Variáveis com risco de vazamento foram removidas antes da modelagem.
- [x] Pré-processamento usa `Pipeline` e `ColumnTransformer`.
- [x] Baseline, modelos candidatos e modelo otimizado foram comparados.
- [x] F1-score foi definido como métrica principal e complementado por outras métricas.
- [x] Resultados principais e decisão de compatibilidade de schema foram salvos em `artifacts/`.
- [x] Conclusão cita métricas reais, limitações, validade externa e próximos passos.
